# CFPB Seed v05.2 — blind semantic judge v02 calibration attempt02

This is a controlled rubric-recall ablation. It keeps the attempt01
model, DeepInfra ZDR endpoint, non-thinking sampling, stable evidence
references, strict schema, and bounded retry policy unchanged.

Attempt01 completed 20/20 technically, but reached only 0.2917
reason-level recall and 0.4000 exact reason-set accuracy against the
development oracle. Attempt02 changes only the rubric: it requires an
exhaustive internal reason checklist, distinguishes product policy from
procedure and scenario facts, and no longer permits stopping after the
first sufficient reject reason.

Twelve attempt01 disagreements received an independent second GPT-5.6-SOL
development pass. Eleven were confirmed and one reason set was amended.
That review remains LLM evidence, not human gold. All outputs remain
pipeline-smoke-only, `privacy_verified=false`, and
`benchmark_eligible=false`.


## 0. Mount Drive and bind the immutable development inputs


In [ ]:
from pathlib import Path
import json, os, sys, time

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    DRIVE_MOUNT = Path("/content/drive")
    MY_DRIVE = DRIVE_MOUNT / "MyDrive"
    if MY_DRIVE.is_dir():
        print(f"Reusing mounted Google Drive: {MY_DRIVE}")
    else:
        mount_error = None
        for mount_attempt in range(1, 3):
            try:
                drive.mount(str(DRIVE_MOUNT), timeout_ms=180000)
                mount_error = None
                break
            except Exception as exc:
                mount_error = exc
                print(
                    f"Google Drive mount attempt {mount_attempt}/2 failed: "
                    f"{type(exc).__name__}: {exc}"
                )
                if mount_attempt < 2:
                    time.sleep(3)
        if mount_error is not None or not MY_DRIVE.is_dir():
            raise RuntimeError(
                "Google Drive authentication did not reach the Colab runtime. "
                "No project file was accessed. In VS Code, disconnect the Colab "
                "runtime, reconnect it, confirm the browser is signed into the "
                "Google account that owns MyDrive/FinDisputeEval, and rerun this "
                "cell. If Drive is already mounted in another notebook, close that "
                "session first."
            ) from mount_error
    PROJECT_ROOT = MY_DRIVE / "FinDisputeEval"
    if not PROJECT_ROOT.is_dir():
        raise FileNotFoundError(
            f"Drive mounted, but the project directory is missing: {PROJECT_ROOT}"
        )
else:
    here = Path.cwd().resolve()
    PROJECT_ROOT = next(
        (p for p in (here, *here.parents) if (p / "WORK_PROGRESS.md").exists()),
        None,
    )
    if PROJECT_ROOT is None:
        raise FileNotFoundError("Open the FinDisputeEval repository in VS Code")

os.environ["FINDISPUTEEVAL_PROJECT_ROOT"] = str(PROJECT_ROOT)
SMOKE_ROOT = PROJECT_ROOT / "outputs/generation/smoke_only/cfpb_seed_v052_pipeline_override"
SOURCE_RUN = SMOKE_ROOT / "run_20260722T135306Z"
REVALIDATION_ROOT = SMOKE_ROOT / "revalidations/validator_v02/source_run_20260722T135306Z"
JUDGE_ROOT = REVALIDATION_ROOT / "judges/openrouter_qwen3_5_35b_a3b_v02_attempt02"
# This independent root preserves all earlier attempts, including the
# completed but calibration-rejected v02 attempt01, as immutable lineage.
CONTRACT_ROOT = JUDGE_ROOT / "contract_smoke_2"
CALIBRATION_ROOT = JUDGE_ROOT / "calibration_20"
SNAPSHOT_ROOT = CALIBRATION_ROOT / "source_snapshot"
ENDPOINT_PREFLIGHT = JUDGE_ROOT / "zdr_endpoint_preflight.json"
RAW = SOURCE_RUN / "raw/data_designer/dataset/parquet-files/batch_00000.parquet"
PREPARED = SOURCE_RUN / "prepared_inputs/nemo_seed_v052_pipeline_smoke_20.jsonl"
INPUT_MANIFEST = SOURCE_RUN / "prepared_inputs/pipeline_smoke_input_manifest.json"
PRIVACY_DISPOSITION = (
    PROJECT_ROOT / "dataset/curated/annotations/cfpb_seed_v05_audit"
    / "run_20260713T145423Z/privacy_qa/full_v052_v02"
    / "pipeline_privacy_disposition_v01.json"
)
ORACLE = REVALIDATION_ROOT / "oracle/gpt56sol_adjudication_20_v02.csv"
READJUDICATION_REPORT = (
    REVALIDATION_ROOT
    / "oracle/gpt56sol_disagreement_readjudication_12_v02_report.json"
)
FROZEN_SOURCE_RUN = SMOKE_ROOT / "frozen_manifests/run_20260722T135306Z_files_v01.json"
for path in (
    RAW, PREPARED, INPUT_MANIFEST, PRIVACY_DISPOSITION, ORACLE,
    READJUDICATION_REPORT, FROZEN_SOURCE_RUN,
):
    if not path.is_file():
        raise FileNotFoundError(path)
print({"project_root": str(PROJECT_ROOT), "source_run": str(SOURCE_RUN)})


## 1. Install the small judge/evaluation environment


In [ ]:
import importlib.metadata
import subprocess

REQUIRED = [
    "openai>=1.109,<3",
    "pydantic>=2.10,<3",
    "pandas>=2.2,<3",
    "pyarrow>=18,<23",
    "scikit-learn>=1.5,<2",
    "pysbd==0.3.4",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *REQUIRED])
print({name: importlib.metadata.version(name) for name in (
    "openai", "pydantic", "pandas", "pyarrow", "scikit-learn", "pysbd"
)})


## 2. Materialize and hash-check the embedded judge source snapshot


In [ ]:
import base64, gzip, hashlib, importlib

embedded = json.loads('{"cfpb_v052_pipeline_smoke_v02_common.py": {"payload": "H4sIAAAAAAAC/+1ZW2/cxhV+318xJfpAJitCNpIg3WKLOLYEJIgviNy+qCoxSw53x+LNnKGktar/3u/MDK/LlRyh6FMFw7vknDn38805s57nXex4LZIly+Sm5vX+JJW10iwuC13zWCuWljV7ff7hZ3YhRMJuTr8PXzKVl9eC3fBMJlxj/eb0ZbhYfNwJlpdJkwmWCLATNdci24NzLsEoEVrUuSyk0jJmWtxplnMd72SxZbpkO7ndnVS1iKWSZbGoannD4z3jRcJkocW2lnrP4p2Ir1XIoEzOC+KTyiIBB8VgBasFGCgB8oRx1SooksWHfWLJwb+sE8VUU1WZBNkGItiuATes3Uhxy2APZ0pU3KkvwKYhLuxTk2xFuPA8b7FI6zJnUZQ2uqlFFDGZV2WtoW5Raq5hgVos3LsdVzu4o338pGCe2V5xTQvt3g94tAt6X5FX3PtXxX7JfpPwHs86pkWTV3sysqjaVxV8hRf4VyVOQGu2o/iZK/G2RHCW7HVZpHL7RsZ6yc6lyJABKX1EXVQXi8XF2dtX7z7+8jr6/ezVxft30ev3b84u2JqB+RdRKKH9BcPfvfmf/jze6F1Zyy/GBZGCK0QU73ixFYm37MnijMs8gpmzqykSr+GZ2d6oOQoKOkKpIwRbSS1vRCQLZGpu5W4bmFDEYrxFNWkqY4n0iLZ12Zi8GVJkfCOyfilCDaQZHDSiEVsoVtZRjXTV0I0MGRI0BaUWnC2SqKpRDVCyKsFm/whVLJIGwZ3Ve0iqYlHwWpYRaonLzJE9LALECpooxS50DY1NjP0u2sHK0CFv6RXVqMiY3nGNlP8kqMhzqVQlsgxJjvRvCnFX4T2eUBKJgEY2OVRocp+YGSbGRXKLjOjTyUdh13ztIRobmXi9ameO1QXy1B/o6bRTZVPHYtUm+qU3iBHztqIwcJLQgwkUfUGtCe/KbCc8WTGlayhj8tkH1ESZKLZ6t34RWBGa1yACnLB/s3dlITraRKS8yfSaXi7ZVqxP7Q5RJF9N34XAQdOvQIscyTZjK7z4Htws7MDhxpknBl6APA7ZEgeFBoE56zzA6vK2j4MCKkcyecr0WnDAjloBjJW+BO3V1JaIqq6s92uicNa7iLldwwB+zfbWgEFMeRyLSlPobObZbwS7LozGB9G1JL93u4yfTOCzfET3tN0orrzSVLBUWziVChGpHX/5/Q/jnYBiCCvWtfevy9OTv/CT9Or+h+8e/uwFg2wveC7sti4d6MM5mKxALBodGxobnp8moOp7LhKO8U8mZ3IB1Eyc09Lu3IosLaosEcqPM7WciWPATv7WP6064GiK66K8LaCjMuDhE1y77djDZrE96LbLtOXQszR2cgkI+QeORHFW1zAo9f7uJHWZa6Uwo/aK3Ts+D17PvRY4Nos5zfo6eo/+IxO/l7fzBfSWU+sgTkxLwDdoOniCrJCxOQKoSP7KKmEAhB5sgwDAYKbkiPUziohi7I6lQX46MX0yA6aoQLoFW00OU6OZuqCtj9UGMZtwQc8VPauqY7QJNr2ezaI/I3uIsK5zWnZgdYyAFzzbK6kigvDJmvNhPYjIeKUvMrt0pNLmXEVOnXXA/+vxaD22boBjRK0jVwUxgDVDSw/qLDVGExauhgrTSjgqGvan9aBGnrTEQgCVL7s3zFyhPjCJFl7LLGtL3Bt5isrc0B/UHJ223eog0ebe2zR7vo4Zx6zSThU4iG7QvFFz10Pcuemjz2UmZjCOpgN7khiIkl9EtNlriiA1JG2R9r3KM441wsU/3N9Ypd/yQqZCzXU3uVuKbkQ9wbmb0xcWzOiLtbKpq1IN+z6Z540mQI8qWbkTm+bNDkoMB3/Q2T+5wdAGgzYzqpuihZfp67HfYxSCHh7r9DJFxNqa72N41a2hOJpCHwTKHSMaEyKGmycD2kEjXE/1ZwMckQDfKkkToyk9EFnfJ3ILz4OZGzpDu8m3HG+l3pm8CkvUDXqRjRfQvIgBK8kGtUsd5yYr42toxigufsbzTcJXjhIVwhP/xenL79g3jD6CJdt4XjAuFqtL2FSEnb7h156jBm3c+k7c2W9+4AylCTlSPBU+jd7wASZgYyY+rQjUOMq5AKggvJZqCX6xHqjgpNzDN/612AerAV/YlAfGTKwsycScTDWMQnpSfvBwXFJRhRi165rvD+VdzkkZC9AlZY4fXB2X4BPFkukGGBs8R8jVo9qb81nGh4x7D7iUmeVQJeFHmSNkPK+O8lClHcj9UdDNootzVvIkcjcy04w2tUURpWN0SZG/coepGePvkOImke1TmJXA8V7llgZIE+Ii53MjtPeYEwGE1ocE43AhDKQUj9xeo1yAuEWkkY8LDsx1a8+p7vWBdHLB4d4Lib0ZU5Es9N176FWYVCJQqqRjbO01Oj35cb4qp6qH5D/lE+BZ5ekbybYbSR96E8KBsmqzbeYE+6WoGtzFNcCNjWC/ovn4jY7DD9Z2NAyk54PXVuctbuJEZIwbBG3pDpO5uE26A2M2PAsfhvk1LnJ8+6DWH2vKLnEHHlF5bR7ngesWbp36DIktbsngtffP4jiuuQgbbUeuddBmzPONfxPcsSl/nCUkV9GVH1exlOtznimoTI1TBCixJgTsW2Z0cB6Decp6LDKFP0G1SbM4W3Gm9z7MYMK2pwDAPFHDZE54kPmH7FOkkg7MZStSX6qC25U56OnvW6iXhQKWsk2zxaAHM5cyRxgYpyRgMUhloh8xGGhq6Q89Mc1n76y9u+Iml80Gb4RBR9xmJVwNwckOilGsbkap/g01/p8bXEF2bfCKbcoyM/Hss78bYV1o0xo3CIRcDl5avji76C50bTZdC1FF7exV8DbFJul+ouR21MXNSYWoe3uMmcw3cEWWGj2Ow9kwZPTqcn6UujIYPE7tCVjR7rAqq/lhzOwjgBx078GB7Plx7Q8Jn2XxhHTAxLp3ZWhvgNpJ0LfcR0PHQUaM1QGPIwPUmAvo3AQxD1MzAP4GXQLdewh3sdHfZ9wP2I3mPbC9HKyRN/E4LBIiGZZCO01Gn9y95sGR3Wfg9ArUpX+3c/UY7bGkPegWBnnacsbeKb+viFy7e+j4Xtev9343cHfq9IGYyhhGo5N1OaUiZ7TvhsHpdrgIIVh25pjGpJ/t/lvNU/tj0x79LxLIEZhfnT4vplgvSaPPoWslztvJJQhxw8IRDh7id6yIUi34X7dQqsn9F1/RNs26BcB92E/istCfInswOnjMbfF/AGZVvsh0HQAA", "sha256": "a35e8fdfcaa7e486f7b28d67d587f4cca8ff76fa5c52eb6ba8806f44804d708e"}, "cfpb_v052_semantic_judge_v02_attempt02.md": {"payload": "H4sIAAAAAAAC/41ZTZMbtxG981egqFuKpGwlvsinjVZyKWXLLmV9SKlUJDjTJKHFDEb44IpO8t/zuoHBDFerii8SNTNoNBqvX79uPVOv3vz2d3X+7ofNCxWo0300jfqU2iPh4QulY6RuiN+9WCz+5ZLSnpTulelbGgh/9HFa9Dlpa+KlLD44r7QKlz6eiN8eTK/7xmi7bk0YUqRFi3+4Y6KNen3WNulIyvX2orBAhTQM1lCr6GywS0Mq9SaGjbrDy8Z1g9UGe5sAbxbaWjrqaFy/Ur2L6kzeHHgxP7Yq+hRPm8Xi7oTPeYVq6UzWDR3cXzdweu9ltfJp702zUW+jetBBNVYXQ/oQyddYfL8IJ/eAxydzPClPn6iJavDUmMBW9gmO9bqlz3KmMT5rTzrwJoQt7UbdOnHW9Afyi3jSEY5596BgO9lWDToEhRgetLHq4F2ncH7FwYjsk8eB3lNMvlf0RTcRcXM9qX/889d3yu3Zo5eLxW63+4QtF/9eKLVsi3/Ll2qpm4aGqP4zOs8/zoYeliv+Mjsa8OGH5R/kHbvROb55a+Xc5SSNayksP8qa8Z62ng5PrMTTrWkV3zMhsG7gqMqp6kK54OXHxX/Z7cXidXmOsCFAAgGfLAUcaw3AkL+o3dWmO4XQ4GmXQlR7GjcZoyOb6QlYu+zSbgNz74AHDx8lnI5/NTg/cIPQVABG+hL549+DBEJNfr29DaonamE1OtnB+SgwFqRSpDFigcTE7uftX3bTXQZl9Z5sWKndT8+ePdupo3epb01/zKhfAeRw+Da/o54AV+yVMwLmfqqPqrOmb2xqJdQ9zhYyvvl8sCYJVhOz0YHzreu0v/BV8l6DN2fdXBighKS7UR3Me+SrIu8RINOXLJVVKkSDgDRwOju0y/ja4difk/Fwg/PmwjEXZO3yga6vTxZmPM4WIi0sFkVB9yP8sZGr9/X4MMjmmCx86nu53ODsGQb5sva6ueerMl2Xot5byihRwSVfbrqkPABrWgke8u0XbKijw4HBPeR7hKM5UXNvTeBs+36jXuG+mSJJ8FmxJterHkw8TVe7Ue+BMo8TTMHN3+HKDtY0EXfBhnr1cMIf073jfWTiPTgORyhAy2sYM2Jls3gBikHKt8Tr8Y2fwQosGNI+RCYmvtTeJTDZI/DWHdmR0FAPNnQlLuDfhk7OtsCW3IOnFhFEnACAv06B4ER5hNhHcciFAgztvPmjkDiI13QqXgZiQIICm4jaApjpmEo2GMC6Z5vVsZYiqDJUYg3RDcVb1B7ZCqcSe8YDMJ0JCHyDyvA3MH4fBpJs54sD9RoJjngsHgbqg4nmTGsQtvNdTqhjAjoAuFUpNc6zfY+yEEM+BdwdvGvBJOvB4YYu9TGfgjcFAzT8TUNt8gwpeb9Z/MAIyQQvTs1qrr2MF4UAVPTkrFhJfB1qEFLc9ZcujF7BPvCLQwlv48wH+HAZk0kgUWxmihgLRmXUMTuRCrfju4mTx6R/iejP3KvyYExauZFCjHwpDJEDAptzKcxp4OV1en/TaFnDFQxrvtYOOBtAmExPluMY0gExN5xDjL1ub47JpZCRuOBIWSOskE/0vBTJMR4VYSlQ3VZ1KAawvkd15+eSlMysJsBia3jHZGsBqULnqZIKI3g4UwgMdVSYh3JzsPGzFAyRY0j91jAaOZgdkgAcpbMQyklWBBBY/Jpf4F8P3xpHX4D2nJZ6ytb1yDNTopRSg7iHdDxSkHxn2uBPmUD5RIhvNDHlgnPywiyQjE5iLpSpwch2JYFGyooiAhFySoR7kJUWUVRzzIzFdZVL82coiDhqsyZ/gzhw5ZLrkpuT+ApvMhjHLEv9BCHJuMDpKWm5ylmLPVIEOIV4kG/aR1aJeEu8boVnurUAUnUGVapVrWsSfyfyMiIUJ2o5XOUGC4AKRBhmR6+5dCLRHNvaM4riRYyetO+QUwEK0Ovh5AEiJNxNwckj5Gfy5fB7zhJU6zhm45xSt0ydtOXQH6lFjnxdTfK7UItFqSoo8cT0MZrjazAgRaOFJ/jqjn1+DN85XrjlQSwXkQDP+pBrAzAogMtWEUtIb3GDCY0yR3cIyIUxBa3WIEhSuzvAJMx8+FGxhsbnMCfSWTaBHsMVT19lsZ8PJhQh1LrluvKnQpFDgBQrHUuuSMB+A0RzWgfyZwN+GTsfVlrMI14nxEPXddj3QTRH+UCrAxF/MEBvnLTQhY6FJ8iP0lhSgRsRYBVJiMdlPXh9zcHjuiBx32vLZUjp9hO4mpEIY+Xjq4hcXYY0RCCBRxTQX5djvrEphKUWb3Mt/lNhnAmcvEr+EoDAdhUCNbBAkWmfp57/QnpmXDwfpTQHtmkSb/UcXVb5yenq8OnzxrpQqUIkH5saf27UG7yA2GNbKzlqc9+7B0toWVvZmBNKqCmvQI8XRPJJEeRk3pO0cPgXXJ08KAzBEK4FOJONxI0VCJbFbRUS2xnJbUchwSWvig/ULghSzTGUmnZh43vvdJvJjvX2qrQRTB58bfy7KTIKR3WSy3XPK2ItbsIWBBQ6TxGGsaSePlCl7HWh7JF8gRyS7MOHqHWel5z0mSk49YVsEbllcXG+6ZJ5q71wZspBqoLKStcc5qWiII935ixgWMMEF3YbCDzl6aoUR4Yzk+tXfpfyxFFrJWGFM1pHQRb3uqOnY1SubhIM2wpXhvykp2Mge+BTRcf1Rnuu8b6o4qu+kJ3AUTSX60OyapqC1PoMp6VoAI6e40FfGvJDzF7LWZgNpcSxXAHdHKX4jPmTKXRP0gy4Ma9RCMFHsNQbrpBFlQpr51IohxUXplNuR7GA47IEG/v569Ym1AyHS7P25VrpZ8k771YgBg/IiZD7qYHlgCCTC2SmE+Gpt7mZFa5/otURs9Nd1L68HIiF+db5bVblW6kBV2mGX+Qj81QW8ZJOlnLd72rPJy+ZoxPnmqdjsvkFHIJ/YUy5lg5SIIEk2VIghqyBLjEBukDtL7MuEAZvyr7TGAu7tkXD7C9XlVhCM2/jxLi0b3LamcLZlhveZoFzdWQh4JAneVWtgUpCSARR0lN8cP6eSRV7chbFVQGKYIZF/wpZaI5mb6TyNd4wANgISlvVVEWRjcMbNlhb54I6PkdzaWxWVOxeqz3LM65SWPf/ovdK2iVVBmijh0suaxzAJfN14toZoOqENFZZd1puXTuNEFuH0mwiezfoC987z026Ti6gaDwkIWrnzGlxVwI4jgL4EFmKOcvqIJou4zEbmK5tXhceXVhpAp+uB0iQIM7pSdIWiZEje2ZJPk5CCfLJlt+1Nq2KV6sqWcsXM63Lwkbq9LqKXvAGiCT5UfZ2BNoSpJaAMO3oM4Ka1fckjZkFZHxRT19pt94qYs/st0wBymI5KXM9CZ/c3Iu5oiFG33iaG8yxF0UxV+x6AlPu0fjR0bn2AcMqKQWjVspXycO5MlNZp4HjRGDWTrrHMdyb2gBd9x/SYX+r7CCxmTpzsn516eP8YpuP+KSSyh3glZKCXecFsRnxKymg+Gum8PjdAGpDIjZoxHqHQnOZbn5VWzH0wFlLyDka4/FByGmr90FmTaxHZ7Or0n+27tFN1mkM68TwI1cnzupcEGeVQGr12PM+VqDfqrhog94YSTWwIb4GyV/y1ABJfshjZh6YMMp4EC6NEKYc09jRZO0s00hMW8eOXl5M04286PplGW6sZFxRp37wPdIRUpkbJ8rNeIAgYwnPVWkP/YUY8Ke5WZedNzLBnkaVpVxhMSpAw5VjGvCU4U8+WJ0TyNDazUY3SJMow6DrKYTuswAcZy82qy8Gfh485zMjsL+Via9MpTA6JwneHb6Pavfh/evbm1d3r28/IgC7D7+9fbt9dfPu9u3tzd1rfiTT3A83v/z6+7u7j7usi2NWa0z0s1nhVVM8DdmR5FIIgNYga3iGkSe4+f9sBjMQt93r0Ll7mv8HzjTlYbrOc49ST0XrrcZhNiMbsySf53WnhMoOOrC5V9jDwgkN+b3SA7IGY9/N4n+BXgiJGxsAAA==", "sha256": "6cf2d11f5775457fa61b4f5679ea3caddcc43f0ed9e273071c2b2cdeddf4b0f1"}, "evaluate_cfpb_v052_semantic_judge_v01.py": {"payload": "H4sIAAAAAAAC/7VY308jNxB+z1/hbl82VbLhQNAqUipxkDuBICAu3EPRyXJ2J8HH7nprewO04n/v2N4fzoZwd62aB5TY45nxNzPfjAmCYLpmack0kEXK84R8LZMVDEWePhOGP2ORLXgOyRCleMK0kCSBNaSiyCDXhK0kgPkWBUHQ6y2lyAily1KXEiglPCuERKk8F5ppLnLV69VrclUwqaD+/VWJvP4ulNOE9kDzDGo99e8BMX//Ejk4uYLp+5QvarFr/Ok29HPB81W9fpw/93paPo97BD9WQMWSF1pFK8hBWg+jHDJB0RKjCSi+wvUoXhYLut473KcFLwBhAqoy8QC4tk8RoUzktY3Q6jafVLCECsniFGis1oPNDQUZyzWPqcHbAKhaAXXP9g+P6JKn4Bb7P+pwFSw0/Ibn2y5DlQqUrRjPla7cr52ApxgKTS5FUqYwE/qDKPNkKqWQY0J+JqdcQqwJPEFcGscI5gpkC0gSSMiJSNmCyDI3kYva+/wItMYIRhTGBG8pJNzlYighgeWX/xn0H4PzO9z8BtC9Hr2+uTqfnszpzdXVnE5nn8kEqyKCfM0lhnwFOgw+nM1Ozz5d386n08/HFxsngn7P/4mHXZBNZYRbuvuRBCXSNYTuznxJtmTsBqQKKh0WKEq9oxGWs4H07uBLr9/bNOspswsjEohSF6VWozaRRw5KQz0jC7QCSDpoizVIyRMIajUSquAYchk1JGUCMlKilDFQTDq6v7d/tPfr/v783cHhwd7RHwG6eHJ8cfb+5nh+djXbRMn+QN2WCtVIFJBLdBck/fMR8gN6SA8OF5QdmFR4R5nWkBV672AUo/WFuwsaNCZOpx+Oby/m9Pz29OPldDb/hDa2zKKl7cSMDB+mQaPh6ub45GKKx+sjLmNGq0IfHmEEKEvwLI9r68a1CCug1XBydfn+bDY93eFCnZLJqAWUSjApbcC0/nju3M4x8Xao8nHIQEseK+uOU9HrYSm0FWBBdrj/4vK/wYAaXh/bhHM7VWF3l+seVbu7dcym2sZynwx/JwiXvlNaDkxn+DLeNI5328EZ4aaDfc+1+lDLQKHn84BI+LNEljTEVqSgYTKXJTRFp7CqG9198tPErjgF/XFDHpJxLMPPiB9Y8g2Dc4OhbdeVG7GpE4ZrCV8uQQbORIEshFe2QDGMhbnj3y0BmmrjyXjDjaiS7JMlsnklMmhQIjxvEYs4FoKqOOSla9EU0A6D7ZL5BCw2bSYYk6Dm3SQYbIpI+Iqtxoi4b69JrDk8Ogn7rd1/uWtul0DMFaZpy8z/6po2hy1x4QV3cHvbZbfCMKjCNtjCa9Jd8DpTJ+vRsqmvyOSfCl8rCRPLhGp40iHksUhwMpoEpV4Ofwv6HZVV1aLOjiLXeaqcrqSCJoFxyiOIJ16a5TGEXXUDW3F9Mxd0t5xeKR5Rm0n8FPLvSfyTSg1pp9MKjZTFD4rUdVbXxabHDXBtAgY144E0eWGTcO+dlzxBUcpCKDA7dmCmaZo5EqPeYExf4UBfS4yxMFEtdYya6rk2ysVjWI+2Ee71I64EpmTGdNj3jlugxj5K3mabjY3lsZeivhd1FNq+2Z7Yil57juemdQdbldsSo5umUMIbq7q82anYKqlePemRaPdY9w5e23pV1WuV4el88UMtUh4/b19zV5wNuihtOL3j5H2JTYSuRJrg/geGY1RXAFLsGqXesV1Ivmbxs0lLvuSwS8sC6/o+Y/KB4rS04osUtgX9C2K9GolwU8t5+wKseQAn/7RMsJ1gLcmM51xhR/SKbomFgHSiItJUZLCptNaUIeXh25Doe8BTTyi4S2eRlsqKXVxcNuwbdfW+F/reljkOn6RiXPLxej48jI6Gn64uNl+r3ow0sGxlA0NMYKJWcZUOL93hoZpwo+wh4TKsxl3bwgeIEDdU/+B1dP/ko8Su4bi3MWPpOimzQoUuF1FLbsChTMWcT1zgsPckaGeyPyDK5OsDPDubXtJ26dx/wEjAt3hecV01fNmHN8UXOLYxMwjVb/FoxjJQBYvBka5dlEiQjcCxXJUGymu7E+KD075HEc8JpYmI8UXgnYxYkhgz9kgYDIcNAwQD+0Sa2GkMY7RkZaonW+Pym8ocKbytyY3Nb6pp/sPhIHpbXz1Ev+2YDfw3HLMD9EaIKm1+cKp4ZZjWLlIzbAwuNkYAI7MhvdHUXpuwt8friTkavfIe9ijXyfhP1F1Tt5Ps7Hg624qodNoFP12R6RDE/1oaBjjziqU5JjT+H2oyIQGlBkZKA4efw7T3DwHtsvABEwAA", "sha256": "cb2b1940a5643f4f44824a5041081e75ca20177b2f1c2fad67e3af45900b0715"}, "freeze_cfpb_v052_semantic_judge_v01.py": {"payload": "H4sIAAAAAAAC/61bWXPjuBF+169AuHkQU7rGjmc3SjFVXlveeOMrPiZVcTkoioRkrimSS5C2NVPz39ONgwAP0Z5M/DArAo1Go9Ho4wPWcZyTnLHPjBSPjAR+HC1zv2Ah4WzjJ0UUkN/KcM3Ikq3SnBE/2ZJHFofjtCwI36RPjEScsGc/LnHUxHGcwWCVpxtC6aosypxRSqJNluYFjE3Swi+iNOGDgW7L15mfc6a/f+Npon+nXHIKgXMRbZjmo79HBP/9nCZM0mV+8Qjia7Ir+JQdxTaLkrVuP0y2I3IWFSz3YyVrtg3lWhXJzz5n52nI4hE5SpNVtD6OgmJETiJY+WBQ5Nv5gMCfGMuDPMoKPlmzBDji6iYJ26QUpPRpyHi0hvZJsMqW9Hl2sEezKGNxlDAqtAdtezRIN5s00ZMPBW/8u1mcH17cnh7R68XhzeUFPbo8XtyMqm7+6O8dfKSrKGay0R2w14BlBQHZy5hdpMVJWibhIs/TfE7ID+Q4yllQEPbKghIlJWlO2GbJwhA2/CiN/SXJywS1OjEL/BbRcRLQNpsTWDbYy32SjnMWstXDdyxqMKBX15e/Lo5u6fXl5S1dXHwiHljHhCXPUQ76XrNi6JycXhyf3lzd3S4Wnw7PaiMcd2B/wmCpY7SQYYu3O8kZT+NnNnQFVbQiLRrRwWLOFA8hL6XW0AmYNUsKfr//MHAHN+eX/1joyWuyTIkDRykrCz41FjSVCk6TeDsV6ueMhY09SJ9ZnkchcwY3l3fXR8D97kLPYE0H/GFL6d5s7+Psx7292w/7B/uzj/92Bt2keJKjUB7Sqfqd5rjVU56WecBoN7ejw7PTn68Pb08vL+o61pyFF+HTNGNJDgtmOf39hSX79IDuHyypv48m9oH6RcE2WTHbn2pPBILAdA7o8Hhxcnh3dkt/vTv+BSReXF1e4zStmfVkQtKcoWFO0K04NQ7ni4vbmx3DteujyGeDuygYxIYDmtidGLSDReURp0ahShhUZkOg88Xt9enRLnFsVWxYkUcBR2U1eIBVnV91mVcgPFjNvNRmmJNdWzET3Deh4X15fXh0tgDelcnmfhCz6TorDj6CwVM/hIFRoLdLMAj4s+FwffgvNLaGpaLF+S/TmqsUX5wVUzhAv5esGOPR4tOlXwSPdIZ/E9Vjr31xdXi9OO6eIgO9w2EMaZSIcybcc8eJkodub9ba7Ivjq8vTCzHNydnpL3+vW7f48T9Y+OcwpywJszRKCgoyruJo/ahN1Zj75d0tODVL92oe8M2fWdK7h2YyxXUwCGKfc3KDRlSICDesYp0rg9oGf1NpNGiQVfwbstci9z0H8oBlFDpuxe1nDDInsEtDi6/ilqcxBANe5OILI7T54tFnRpfbgnEIF0kBk4kAO1wzbyZdrwwGYkTVm+Gi8sTLnf/cz8Z/8cerhy8f//z1j5ZAv6ICZFZz7ifRivGiQ7SN6qLgSDnY7VwnBfcO6M55gBnFD0GsdNqi7FO/YGGieT+tinVi1ZAkldyaJGTPLE4z9ERUbjsFIwshp6GYi1GwNqocDkjXnPYbR1uCKOqyCMym+VmWQ+gJYePq+7KJEhqzZF08eh9cy5ISf2NZgFxwFJqWJdgfLfPYtETJikHshFgjrXBOwLUU99A9wuRNphJ+HKcvIEbOfDBtoAzRjOKIC0JJY69cOc5OXks0YBHCNYvKpCVBlsZRsLXHLtM0ln2Q06r1DQaQ6AANz2J/S9HWh9LgMUdwyfhvSKWOhUwTQtAekjQyjirBlKSQRSfVCCCNYaOeGS3Soe3qDQ934nOapTx6VfxUVvgJ9pjJZLDJXQiBO6UWsYyScFid3RFprKNSz1xnSKAFySTiQpFD15rDjyBRQvJaRiqU4w4sKYwjMWNBBg//MRkiDvNaWnatFLLyK54QCc8TJGRw0rGnmWp6VsZpc3KVKlbCj8iTKuX608gyZRXTjYJMp8gcWj3mqLW6dHBvtsOJAy/eapZBuNUMEbWDgwqBLeIygZjbapZpfavZJIStVbUCWVtckea2mpsOpdZKpfbn4ryNBsL4Orx7zQwbQxtmaA6BMTLhJlUVnLPfS6iSoKZ9zeDQR01+3m1eQhm8AtcMtM8Re8HS0krR9C46FXu3QzpcLlgklI7tk2KJ6FjkWGgr6UT0bdog+BKM8pM49UM+bFkn+Ac/pAUE8SH41hT9v+eUxWr8k+O6Dcuss2pY7NuMOhLWOkfb0N9mB5qzVyPLvSqEBlgI+WsMFpssZgVzXNQU6hq3qk+5wpJqe6eZIYcoMRx3ywFmAiU1BqL0hcPUf/DI3gzr6jap9gk10rflWzUMU+A0SjJgMAZudqQD4KZQAi8hrcUQ0xZF9Mh8GIMs5HVB4YzIl6/VSgWFJMY9gwVKd2O0e+Jj/QsrbZMGPuQXCNOYCqox7u1lK6MOUyaHiXOgliQEB35pLHOWSmpjqv8/K/mknZ59QGI/eOJmGyqwTM+jRAICCKWgEXMMmiIqL669Rs1ZRDxKIHpBKjRschqJZMRF/Te7JN/3GtlRjylpqSF02K5HSdhxzL9x6rbXJJuSF1KLmOLhWbIFkpzfFKDSiAlYlX5RrKbG3iskmLWJgZXIkIv4aziku/dul5zyYCLMY+RTG/vmARnjsPfsljJEVc+1fYHssk6/HmuKwC+mmjBJvWMyKUcn8XabTvXtNu6DsWHlAacZEy67D0tVBCgAM641pxnNmg1PdgNWHjUKyEG4qB+gzvHjYmv3wcoBPJbBrN278V8hr35iCa/zS58Bacspj8u13SGKELqC/yzRH9TnEUqEEJeDriBbqPUKxCNI4xhiBzowqwtggTofrG9QZSDtMmbhjs7XIC7DmuIQ4WiuTQECLUmxDEbYF2tBzmDfQ03yVWamEeeY53iEg90wqA0aNjLGsDOUHyZsq2E9tryqe3vpVmXdqWy2lAdnTr4obl8tByAo7lv78PDuWHMip5J3HML34GkGPRO96cSwbc3bsckP7w0r7ZkVN+H31KZAhWjxbs3fNKMHdG5Q8CdbRwYGSYYm9R1ysQQgn4AR5DvGKcdmSuES/3187ehKU2VDoDmIh8V2p/OYEwfhsanAyCYHY8DIxoCR7T52c1wYg4ucVe53upk5aWT0PUhcxYkiCufUxyWQED1GyROeK5ko1bA0K7G3jwhkUKUf1xb+xKCkkZsgXC18uwR0SeAH5DBtbX3VG9xkBtvaIn7HJlYyq+2s+IXRCjAWLi94MKGUpQyYW5W6NXfU0kp9U1s+XW0t7CpNVyuooXAlEAvzMiioRKHjXY5/TmaTH1veH1t/aoWAOWQHrTCApLO+WDAnHyYH/QEBSWa9PnguPUuvK56Lo9ZhIrs0+W5zsRhY3I3Z2BPYlmO1f0NZ3LAk4RJKzupm80+54URvOGmfqXElk7YYoi2mUSrrqrWCEgCi6UxdJKAvUxfRYvajA5CQQI8NcILKOucCpdmo0A5wozdHW6gxpBrTPnS6fAjV8QQwRudrAoyENd9XUwhIDu7ylpDjUYkIwcItaMhCwDStwHbkvR3QWlhPi5YHj1DCGFoLAGrRmqS6Iq8DQ60RlSdvaxIG71Jvk4sN5wLIhStSUFcvqca+hLYsGKx3kKrKIGDSxxJqO7pOY+RgYW69403dO2pgga1hzctKNaIG3vTOZYH3oya82Duwoy4BDjY4o4ZLpHvlg1GGVQ7ZeSLVXbZ1JJ3GsDrM0Oi06scRmVkHTNGhTHjFiagvBJRh53AkguEO4EeuYYAHagJwGljbsGGVdS7Uh6tM3AVrTsVIXxmBCF13TGay6uLE049VJkn6MtTvVSbQ4wJUnoJ33yA4bXbJAvy8DqzQEJpcykNN6FzPSrEeLGqdJdVoq9TJptSlXI2yqu9syuY9jWcnPPfw+8GOXt2Fg/vVWnrHlY6nhnW+GrFk6bjq8b7UQ5AptiE0d5j+fVc5/jCqM2kjC8isic1UY6zVWZdMnvjXutEQ10tNeXedVLUCk1xUI5rXeV2JiiQ0Lq2bIMujZz/Y4oVntIp28lnC7j9u/PyJwg36OlqK5K+LUNh5DFftMThUtc9tUktbeKvmNVKR28eIKzRUoqC1l2nVqzQLYCPHHfBIIzPZgLWVWHUhhCMIXyJwMr9c3Y4PJh/HN5dnxH7dMBJFlFAgpFsBVKfBdtLkeZxKrD/GWwIZokfywI6sem5EZNTlI4Gion0QNHpxanB9Tnt7zbu7v4q3eGIKhV3DuYE7L+KThL0o3agL60mrZnGbVzLqvdJk8xRGeDUnHi+Jyw6IKa9wJ0rTJ/HZHvmSw0W1RPCNv0GwPyw3GYD9yj1OpGvCxiH+9BzxKMHFBAB3gPo8iCJP2gR4DKxVvD3rjDevB+yVqBtEPZe6uxPmu1W37NJyKnFo406zfi88f4e/VwtSngCwbliO0UBtnr5LDmsVGi8x99iYAsroGzz6yZqFnX1oMLAHG/SzlbrtW+1Kqqq8az8UQgYTFNY4eIlkVmO6Lnfx7weyqD8ilHmhSmIlHBmkOfbDNRbGEoxPE/JJeBd5rhoMwb44xGY4Bn6h3ySiIDrTlChJhuYNKIlIs9Pf8G2jjPM1brAMZ+pIUDYxy6yvQYGkRZSUrNah9kSnDtVwd9A7VMDTWm/1G2gsLwQbc1GNDsCuOKqhriGWb2Hqs0qb2CEbVL5lXNSLdcuhoz6GGthDtEgyc1vYinIiMKSyrVqHjTmKpys2pWypleliRqSRv6w+JQKGVPlrVK9vUWS5qnuzkod3wIvqDu3ZMjiV3gGuKDlqWNG8t4BG5UfEE2UKb5X5UPgK/Wp5cgF2zDM/UIiaaMQnMRXBYb4uMahciZ4hZDTiuTDM71EapgG8GbVGTvwwxGnEkKEzHgs9j6u6AJ/WeuLqHBKelQ8Cel2vIt9kqSuTN/iJN5K9zGqlRw83816yl50pQnp4qceSvYyq+riHj3ww2ctGXXn2spFvI3vZyGK1hwe8jnxjOVUJ27sg+QpyNytz0kAogQZYOJzhbGXVcobGI2dMkESJITAGat61SbS04zFptnVqicdbwkn44XuFe8fL9W+WzOT93ylclTb0SfnN4mksZWwBLO+Rc/db17dmx2MiksE3jol4xtrLRpe74+UWsR1VKFpJZ/+4sXTyMNQXVxMeRCMoDCjgzhrM3sFApoljkeZiaOrhUL2XE4zsuKBCxcaPEhkkLqDQl2EBCcRLP4taI7bwNVFpqp7fxDMow0BCK5fuSmgFC7kFb+XSbuPdXzPJbb9163zo5okp7eY60GDALkMp2qxcvg5TSTrTaMEcFholqVpFtoWBSgpVcVUEFmgnCWTDyMobXqxe+LKZW5ChZi+brPEGWVUsRIMhsOBUSSAbDEEdQJU0VZtdAnVCpUp9rU5LBaZe8yyD2YE7IYHV0KLSb+FsQtVWcxdN8/3OUhDPGP7fOMKDwv9X5sFlJ6V44ih1dMWGx2/wXz8hE9LWNgAA", "sha256": "651549e7f7c1f130ba9729f3fb5e21d0cd82b84afd3bd9b88bda831b3410ce72"}, "prepare_cfpb_seed_v052_pipeline_smoke_v02.py": {"payload": "H4sIAAAAAAAC/7U87XLbSHL/+RQ4XF0FsEiIku3bLG+5jmJLjnJeWbHlreQoFgokhhJOIMAFQMlararyGnm9PEm6e74HICn5sq4tLQHM9PT09PT3jO/75xVbJRXzEi9lDauWWZHVTTb33p6c/6v3mbHUux2+jg69VbZieVawQVnk9169LG+YVyfLVc6iXu/iGh7KdTVnXsVyltTMu05qLyk89nWVZ/Os8dZFxW4zdgcAV1V2m8zvvTSrV2WdNVlZRJ53cZ3VvRXHpvKaa1axRVkhwMW6ZrUHD8sk9+YAvkqKOet7t6zKFhl8gsYmMBg39bKm7sGAKaOmFVuWt6JlWWVXWQGgiqSqkia7ZYDlnFWrpk89a1aktUfTTHopW7CiZoOsGKRs1VwDpDSZNzCLq6pcF2lWXMneXlMS/CtWMIQLiKyqElGoop7v+73eoiqXXhwv1s26YnHsZctVWTUwaFE21KHu9eS76goIUTP5DOS8zrOZfPx7XRbyd1nLXxXjQ8zLPGdzAijHeAvIwvL2YZUXyTpv0mze8MZp0rAmWzLZUj73Pfz7a1kIoKukQQxks3N45B+a+xVSQbw/Ku7VJFZATmSD2lulvV4vPv/08d+P317Enz5+vIiPz372xoB8xIrbrAIWuGJN4J+cnr07/Xz+5eL4+OejD1YPP+yZj9A56HnwDzEJWrDDqGJ1md+yIKRW2cJrtaEPLAdu5TDiRZbDuhhdI+TGoqknL6e9sPf5+PidHNtCZd/zgWzA9s3+fA1rz9L9GrZOvCrLvN6fL1azGPlz3TD+HrbUoQ+zOf356O1/ObMx4NKLDtgGw3DgEmicrNOs8WW/al3Eh8PDPw+/O3h5cfDq9avDl3/bF7sv/iXZX6zznHCBP4APTPCnj3893jTDct3ADOp9zd/7JAZi3Co2HoexFBcxbLuqgj3g994dnxx9+XARExVPz2CNYRBNUhjBgaEHirMCho6Q63MN6OTLhw8EbTscXMNf1qxxMPjp6Oz05PhzCwk9/jIpsgWr+bi6N/Lnx8+nF6cfz4hKxiJCdzVxSWdDMAHYAwfYxy8XQAiA+Qnx0OTH1WNzgIVLsy/EYsrpUIMo+aOHMndWNg3swIM/eeWCZM/L/nD4yoPxmgEXVCiFhHyqQS4UTZKBUEgabwltvINDAFSUxWCVJ3N2XeYpil5YU2gTlLOaVbcg6ZCFBsPvBoeHIcjpT+Vd7c1YXt7BiABohmIwqUBYgvj5Iw6Wr1PoNePim3QEyIe/EH66MQhaoExTZbM1l9kk6udlxfsC+EgR6afTs/j9p49fQDScvY8vgEhnn4FcgP35h6O3x//28cO740/x+dHFxfEnXJOKRfNyuYLNHFT+5SR4Mzo/PY3fHkH/d0cXx799On539Pbi+N1vk6PB3+LpXng5BeFCcDdBwZbJ4Nfh4PvpHsCb/NP//vf/DKbm2/AFwOidn54ffzg9O45hAM4gmiP4bklymFus1aHf+3T8H19OAaf45MPRe5zYA21gn7RHDcRJ8nFTrZnf5+9noNOul0l1EwPcq2yWs/EiASkmv0vOE/oxVV8fYaj3x/8Z8/kDejjYhDrZJJvhxGBWUfynvcF071/kI/y+jPBh+nDYf7yc+ahZo9Ow34YRvPnhD5dpCLS63HtzMBlEl/X0TfgGn4M3l+nDy8fL8I18Tc/iAX6/egzeYGd/C2DqMoC/h/R3a5fL2XXTrOo3o/39y897v13O7u7uLiP4uRt/AP1934A9BT0GKtSrr5PD138mhRGgZhyRCgm9wY8eMPWI4KXZFUgPILFQ3hHvJBTSXQbWBHaNyhUrAr+a+SGqymvYGjnjEPAf7CJvlpfzGy8rwKphVZAny1majERLUFZJGhwMD195Lzz8X9j3Zr4faggal2i9QvUeELxQTBqMkUJ+v2Zf+S9A0ppow742wW2Sr9kIJ2hPVMBwpkmtQbvPy5QF/rpZDP7ZD8OOIfIySWOS7S4pc5AQEzRVJjBWH22L6XTURT0aBaTMWI7TRUmB5gSHinDQOsB9GRKJ8RdSmHdCcwHfRCihVkEol/2ughVo4QpcBEJx1IkuzeMMrCiOB6HMjYpoeZNmVSAsjPEF7PA+iE+AEZc39NjNKHfAte58+17B7hDhsX9ZbOYjQBPnSNha7CFYiaYXEH3S9XJVB9ASB6vRXk3qeZaNT1CW9MHar5r4ht1zvENvz6OBBZmE/RRLAzxG9AFY2UiCFcmSc1IfeXo58myyEdWwpWSwnBvqY+wSYA9uLiJcn0ML0WwjPRb4l5fw0t/3Q0VztK9wfFCsEliHhYgt9FvQjA2Si1a6770w1q42tleVZGBB/ozcflxVZRUs/HeGMyJp4LF6nqzABwHPxLsrq5t6BbiOvAeJz6Nv7UgcTZCTBPm9tqkMkyIQkkbbGAZb0jeyZ6QpY30lIjdrcOEcpu2DvR69A6PzpALCTkfuGEBMYw+5g5NA4gKjtS35DMEpZORDSVXH1ZatJ9GqBHh1DSpy5NHu0G1hPuskj7kOjVesQu+QGnL+1C2FPxorv1HDdJu29WpHI+6GKutufs3mN3EN1vi6htb+PC8BeIx7FuxloYLnZMQK9fxIf5dZvUwa6G0TAXbUyDMpSmwOb0GokzwNrf0MH8R75FNJ1gj3Rx3opsDZXSC9P4x5ZwMvaKpR28rj52K5LMZYleDt3ysQwN0amuRvGAKcmdYkYaV+WWdo5HJHI17kyVUN+3gyDQlV20zagls3ahK8R3DJ3gTcanRdgZLrQlIPhRhJDbHCJDXGrS0mBAHnZ7nFxy06+/IbTOThUW5wMVGjn7kPQABlaVbGKCDuwcSgKEMtmR38gWhe3/oGTxbsiiRIjAZ+VeZbms6vwf5kxRWDNjXEAequRrNkfrNeKaFhMa5YPYk6rUxrQs9eHEWJrLgFqpbgIWQgKQu0x3KIS7WWyCQ7rUdtqhCUb1OkKscZtwqqCK5raKuIrnKraISFsujWYCZP9A2Q1lZDrUEcQrrcAq4pcgKG41nZnKA/xAmDXSw4rolJpNaqj3/2O6Hv3qiK4Gi0mfsVJyV3apvIE/yMlOXaiYgr+BNfplz2AzsFmwA8lb2nmh9TiDCxMS6sISfZKhahrLhIxo6UdvTOoM6uBBPzickN82Skd+2w3xVbMk8T8In73qIinpMbIUZLjkwUBSzwJUVB4siffe/719+FfaORnJAnJlSTCSmo0geXfyiaG/yFNjEYoIQDMaOFxk4+/KL2r/dA03kke3SOkUlgPA3aZD+xn+jDRKr9VZbFyEYgKfxpxH6B6ZR+GIFkC3ZvBzl2WhJk6OTNIRB8T/EJMA3WYEfAF3O3bMRHGhc80MJxcSIAz0UL9RKOgHYIxsrNsPsuurTMoSn6L2Cm3YHXGBKheCjguUihIK4gOtNATHyeJ9nS4yN6fMSNeHVYVL8HSivMW0AkSo7mrREgF08JRE/uf2UV7txFdhUL+W5pdwi2uc1wY/NIHdnQ8wRjkEI1bYLhtLJB4Camfrhbu5Dqd41i775/WLM449p2fBdSu215rbAtYB1z2Q3LdlVAwaG/Z2jBtiujLEoHeaEiKSVlQSSVibKrNdZWm0VkrOQonqAf2Ci4JhTVlOkfI2emDCiFpqLI74Qgkt22of4h/FyWsPAzFqaLdcJvwFfR17JKTPSawCUhQChQBVHsGNXY4ID7Cy8Ph6++AYc0S4loc8xckHp42QdIKrEpB3LCZ0pZaK0rI2hNAoIPgyUBOKBpuaTUxAgnw21IEQggy/V5ITbtDPoPBuzH3x4UXPhNMB99NxTH1XtXyK/BrAtGjmOe4eXjbI5xiUBD9ivjs+LN3bn2doT0hIA7GHo/jAkY/v9w+CRfwsxH877LdY0aAf5r7hgrECzmHA6HmpvQ4sBJhd4PHPlt7u5HzAc/qC6PntRoRBdS28ltkuW41mKIGYQHWJ4Lx4QHWmjS8g8EWrrIgfa1kaYNsI22BDtjeDiOSiHKfxgqg2bSu+amSg3ZmAajYyB0+5ubQ+4hXc+f0BBcaMxMOM20QSCIMME/0yhZQQgzxd68Bb5FIxzDiSwN3F6mCX0/5nFvj9O0e1cBHvgZ8fnNj/5eZkWAzxI1TUUiGJCREBhtwBaxsklqoAFz2IIFlxJ+XxJrwvOKGdhkYeiQqWY5mcXdmwtTNFO5o27gacijwtcgeImFZfc2G8MiXoGZXFOQiZwOK3q0gQYyFoqj/TimISyyOFYHLSM4EVmxZjabCLzkohMvW5AmOIZJjxbSGPZzEbPnPB47k1ZLCebGTct6UsDtDnZjmvoeJBhN0euyqEShk0e3MwcPcG1gDsmpMpKOWdxYFZvEIpnL8UBjiisO0xXn7pgVw7WCvTzqXjRC7oKaKKRg5w1IM2EDg/kM0WMOEzVlTAsLVTWgQMe+Uo+awE11P3IESFI1mFfAJUadzmmAb4ECff0KWsAL7SjD3KHOJvgruyex3PcuwNcWP7W0dhi0xZzZgmMA+wV5CYkYknJAhH70uKKQLeCdgzySS/J0sEy+BsM+bw2YQ/lUoGDSDMMwNO1/yN+hseG5iVg9BocP/QV8MIAiAh8gfHrELyFPW9ELBChgR7Q4mCUkFEJTs/KFdtNh2K4vhAofm8Qe77lk1ZUSTPSngy2M5QREnFEwoExATKqOxbvJ4GA6OZjaBLY+wUhIYusdp6vmCpBrXRDkGk00flNptIkarjHNfsNEwLdlUIGVisFNe1b3lz8nI+o8xUTYxKo2mPrwTjVDdpqaYkV+6RMviqHMxKhyAciKj6lmB8UFlwDqsZX3kQpf9ndTbEIk6JdokPA5QrHaBxiagJMfQmQSJXGLRtjnlPUcUPhGFM5BlH+ZoAmbpZTxiKjqzcn4uHhNZGGRPxVyUNbrTCXz2pOM0LZFA5+KtiTkCc8FTkP6vM1oPFGz4v4495HQNZIoQakfRDkwV0DNzgU+On1huD42blZYbCJjtNOnoSPG4c5PWrJaBIJwh2/22ijUYsQxBfUcxPpYFLheFvVYK5u+KHbCxZVithXjQ1msJ4Qi35+Gz5uPivK5k8JMaNsHFbEjpRKjrC4SiBUlxT3kZGGbthqkaywyxdI42ezZBDdzQQqcJ4bgXlnL5QOt9ysUB7TQSWoMAQfow/UtZDmhze9haKd4hXaPy8UCsiwYQEmqLCmEun9hZHSz1FD8optpCuDqP9c8gLVWAuAEfBmPZzMh85KDzUQwByQJzkUM3+N4cvenThaMxChuJYGRlgAyMCjx4jqtxkRpIBpvz6m/b9Xbip2bGIhBVHiRfYVosqCRDCXLUihlPiOegh6/j0kjILpmjXr9jaYNVnZA251h0tOCxLCaI18omrZBG48KeAFel200xL2AE0J3nN6J5x+5iJAruTtme2IjAesGIh8YiDkapr1ulFCB/J/QHWgvixlx71N+41uLe6HmXBSS3AgYoQ2AUk32ezLqCGNf4C/jUi41FSzFba4PIT+ITa+t+gVLsCa8DtSWdZQzamy1Nz8Jyw28ipqnoWFFvoLIAlquF1BvAHwLgS0JtM+tP9DNBVb2X63RAar11jSKP9EX76ipJKsS4/O03mJGsICYAujuUK9nge/Biugeoq4UnDqz0FKBlgAtIWukxTWxCkzBa5xjDhiqL8jypgcjFLGlIyTDK9nPF+EC0b8bgNUZNZrobFJwd09QCTxfNHICNi1QkFUI2i/3zImGrntsrWYpy3mtVlTyfhANdRxCZvg5X4pyY1GIw2uOOzQQvjctzs22qBXLNj9sK1wSZSBQI2eBohhfvDPYyItsCsNz5oSgz8CE22qLLW+Z16vxv51xSxkYtvMf5oyfkEVZSDBWtI5H2n33LA4ErFM0saE+aYudgwdqPHWgJpmjghHnWpw8gIbXjU1HGi+U2pfCStvQ6BqHR2fhUA5Uo+u0HRyy0fAtt4HzWkyGsfIgLMfBOCvATwlMtfXNU1/zBlOkFqBWbom+6pxSq4uZ89avt83+vT4MRB2eYdy7G6RvllpsKwl0N1Xf3n+hs1khJJehct3ubbruhFiEUKUGJAReHOuSU1VfWaVOyhNp1+uJOLVZFiUi0kb39SzueC1C0na79kscWx0Bk5a31YJn5+Re3PhFlv9tLyU0p7Ke5aIosPP7MgFPAI9rYXUMcKA4/+MWEJK/omLnisIDKtDCJZkMsXSO6lphfUjqO41Do+gPwW23wF1eFg5/prwnXu+Hv6Q5pLIk8qCBYJdNZcmm8aZyDM5CUNB3i1BUMDAosQGIXDMOS5+JijFkBPhvALJJEtrBdatL52Lv6rSJA5x+U6sCS+3kByNSha4LOYuoCFc22eXHPv9oOEJ1RG9m9zpxpFr7vPqbVzGFBkuacqedvHel0jek/EU5N6i0qpzzKP6OlInwoxWF2n4gt5HjzclNJ9YpuIhzrFxPI3rLT8GOeXhAtw6NvAGPKXBvhrcwEwCWKybaikJ0R2bvdF6MDUsI6ygH+X3K+er2u6Sb3YFoW2waeJu+NHS2uZPYWwymFVqUwYGUyXA0NMBsXLq98dOiJQ7Bx3JU2xTmIMbyBLD1UXmPY4fyEwFr6rQ3Zj42H7oSozrwy2Nj4825HnU82YRpEFz4eQCiw6OU47Ta82CHCJLL5rb3LF5OdjtfUwgYdJrb21MxkqtisVMUO3ljuTTttnJGVmv5st1co6SORArnDfvRr22dJBGotXwwUqxaGElaciDCgBM1CeOOqgqzb990bPqmP6OFHkakd+aI1RQbOCKcW/EuYB72tS9FFIqvAood8UhxwAc3IjlPllJdUiJalaugtaZh2JVAafVqra7RT+4UjKRZnTavcdi5Twzp3IKgFtzoajbdZDjSKS93jnol9lxeQ5Fs6mp1/kNQG35PbfHCT3zY9b5bbOmdNvUW23qHjb3R1t5qc+uP4lBwTNHxrhasqDKwtMAGghVNanV6xmq0BHFy3T3hK9glGKRRRmFHK4iPtk13W0gLQSiozhdGfXw0l1IerbS6PnRMnWIqYrPTjvRHYme2GxtebUsrQDelQjqGsfaeLNwbWcc5Zdi9a+CN+2nEubjd58ULFW9sw3M1tNLmraNd7gxaBqKYh/u6i6u3HcVXkJyX2+Dog2mb0TYObW9rZJ/Qbp8ze9ahtO2O5ebmm7yMzT3MI/uwuaAUsXrKaI8d9o/UaFZNmBP4e+oRWbGdRF227g/x0wUcdViWXTdT8NWkkj6JS/go75ggF9usNHbhulzBw0HOlRFUqGUcGTbw7CsChNZgbpreDpgolLA0gYeQfby7wzzhBfIS1ee6mSN3i9tcoqK8C+SFLhF8CyGtWtJhRiwsMWMTFWwHRmcZN1zpYWyDzdGSkQ7TyVfTnRGUdif5paPv8yWD7OleLKI7t0J+7c7aFXCFalc9QL/3/yWOlNBQtDKaTraeS51+28lZbYPiwhgVbGa0DhLotTwChKQwrFgz1pbrg0IyZyLZv6udMGaw7cQukDMDRhKEOT8piGiomF9CYijPK4qaEwKyZbixN9W6dHc2bffujI9obJsAvtCRuL12+lS2DPXxaq7legnMCTes1GB6ApAul8vpJi9kMWmvpg7Bwi2z4fJES3nE+gAC98A1cxDIWICqLnSBC2h4rfyWC2iMGT120owmEJtKpkVB9Bfw8PIIzvGWibiTIuyyk42T0VDr8BlULavbdjNnryf4t2YBpOFA2yZ6GKVwzB+QB+9OEQlqbobR8ACK7KLha/z7GqqUZJFB2DqtbZLGtZs6GErGKGJe+8A51zAsNrA67yziCaKMw+y2KfziQoAykZiXibTMne41VmxRKwtWOUktUYBfIPhF++UtHshIruAoHNVlMKzXhIvcGCAGES0ueP4iKkKsq4VAQGWg5ljUkU5wmY3WxGY4HjEXF6jxulTpEHVKI7WioZtfbTtj3za87c99CxL8nH6bm55q6u40c59o4j7ZvH2Gafs8s/ZbTVqlcCF02fBLnbomaW1mXvbdJjoWLo5MI5ZqGJ3xtqtOvme6DBIDbKs9aGkIaDasdoZHuwEkE9pPliJ/tDJQpnnMr47hfqVqblwkIxt33yaD/i9UWh227pXZeEjaOiAtK/RN69rCTj/KCge8WzGGSxZhh2CqX163GJ2p81ji6hh4WYENrhocVVdrDEuc05eAi/sVMsw4jtNyDtcHGj2jJE1xGOoS+IMB2jEDMi+xNgZPiovKAn7eZ9y+JG8rOKp8IwttKzR1U95u3PSlDzvRkzfobQUq5MDAPE69FbRxvd5WwLCbAGi1HZi+Xm/7zIlzBsjsEh6dgZDgDodbu3Pb2FoHpzvewXh4sBmIFhKDAdh0A6UuBy1bUIE3CuTsSXeXtLS2i8DD3Apid2BZRuBcXIUN6N4Ho7V2vTfuNzomvamayC4lGiPUSD/rCdo+FW+n3vVtUHJgA5qSPZpiTi6SN+7wy3QX7fzzxiV/MEbXno8Yu8sVMtwo3qrTr+oy63nzzQa/zBNkRbcE3hmYJHT9EY/I83xAZ8xIyAfe0D6N3RWK5WpLV/vbvuD0acFHG0Rng2lXSFL3ERaP0+rRyQa2dZMdCpZ6yoln4cbBC2Zj1NtwsS/WFMQxbqM4FjVafE/1/g/yZUG/clkAAA==", "sha256": "d04f501f1a25cbfd13ad7e14fe088652150b07e12d8a2bee49e6225bf3a4a595"}, "run_cfpb_v052_blind_semantic_judge_v01.py": {"payload": "H4sIAAAAAAAC/9U923LbyJXv/AoEeQg5oSDZjidTzDK1GktOaeKxHdmerURRoSASlDAGAQ4A2tZo+e97Ln3vBkjNzNbW+sEi0N2nu0+fe59uxHF8ua2i7i6PbsqiWkYvXr79Nvp08jx5GrX5Oqu6YhH9uF3e5lCnqbe3d9GbTV5d1tsub5LR6D00bLZVlTdRUXU5VK+rrCzvo7usjao6qptsUebwJ1pk1bJYZl1+hODWUDXKmtst/kiiiy5q8mzZjuoK2hbr9bbLbqBdk32ObnOAniHgdooDLZpo0+SbrMmX0S0MCcBWt8dldpOXUAF6ibJo1dQ/59Wo2d40xSKJ3vAo2kXdQN2oaKFKCUNppjBJgAQ/o09ZicODkbZdvklGcRyPRgBnHaXpatttmzxNYWSbuoGBV1Xd8ZBGI/muuQVIbS6fAQF3ZXEjH/kPvEjWeZdBR5ks+bGtK/m7buWvDYxvVTdr+dzeq6KuWKtetttiqX43JcJv8p+2edvx2BHhWF+OXD5PCcrPdZVzvU3W4WhltbfwyAXd/YZQxu9Pq/tp9AIWGFdnGr0qAIdZKfC0uV8yvYjK32Zt/n29zEtoUler4vasWHTT6GWRl8tptMI/qcL6NFpjVf1i1DX3s1EE/wh4u2iKTdcmmhqSKl/XKWIyXeZtcQvvk8Vqc5MC9T5NN8UmB4LO03Zdf8zh3dN0Ua/XdSVHNybY+O/8U7HMq0X+bpNVU/X23fn3p6/fX7xIL89P3715nb54c3b+zigWzPGdIGZdUtbZMm1yoLVl67yVHJVKFjAqtHfZ0+dfp6sCEKtefm4AwylSSMkvJ6P8yyLfdBEgdlvmr+vuJXLAedPUzSyKfh+dFdB1F+Vf8sUWsYSsl69v8uUS2OVFDQuH/IqLn2jkPgZt2AkQRT6LAOV1k19V9RGwYr66/n+K0NEofXv55rvzF+/Tyzdv3qfnr3+I5sCJSV59Khqgs9u8G8cvL16fXbx7++H9+fkPp6+sFvFkZD5CY6YtZKKxB3sCDNrW5ad8PKFaxSry6lABiLNcwKBJpKnRNEH5B9O9enY9mozeff/m7+eyc2ssx1EMonqz7dpjzTnHvLgoa49p6ds8XzrrX3/KmwaWMR5dfngtQRv9AGCgo/TpydOvT/789On7J8+ePzv5+l9QHfFzcXb6/uLNaxsfTutcsDpK0WPF9khyx229bRZ5Gu5gMjo7f3n64RXM8PS/ALgaHwLNPh9bEoGe2rw7BnyBWOyOEJPt8U3WLe7SE/yXiJJYgX17ef729PL8zIEtlU5aVIRPEj8BzDFyn54kRGUm2DffvzWow1mmBYlIa5lI77bHmjstckfufJKslyZG3nx4DyRqo91bEHoLPQrwNejzhvR5+tPnvHqWPk+fPb9Js2fY6ZM067p8velOnh8vYI1ueGgwPbPb74GVX0GHMQI4JijJ8yOAcgRQNAa+PX13nn64pJp3XbdpZ8dG70lWHGeb4vjTE93iuw9nfztPL87UZOKB0S7zHHTVqsnSn5dNWtVVd1dUH0F7pWTa2LMxhw9L8cPF2fkljksB0YN4d/r921cXr/+G9V5evDqXE4U5pvVqVSyKrASiaLtmu+hSXj9Y9zdvz19fwoKcX6b/OrsEtj57++bi9ft3h2DgOK+WmxosqvYY5oJM9Y8PF0CSCkoKFHr6/TkAfwew2NwBMmckPShxB1yWARUCDmItA+N19iXtgEir1nyLiEHKA0vHel1v0o374qMFrqjsGsApLcr/FGaWld29WQZMBAYI0VCgFNnJrt1uQDzkKZpCWWdVJXRvkSGFhBOluxHK9EWZtW30rgP7ryMjZKzMkQmbFWxvMNsBDrWJMs6/dE02j6HPm2IZa2hvt0BIi3OxNB5AsBh/YDEGqrbd3sB6RPXKsJf/0EYbAhHJ5Y2kMZiQuXnQsFjt4rCwepWtQRUDMozGxVK/2TQ1amPgB6til93qhxZs2W07Qwuenn/aopT5mTidakX/Hb0GUxGGg3+4zXaD9gBgH+1nmEbeAISyaLsraHAt+gbeqKCjJQwfX0/RgLwGMGQDjsFoyLZll66yBUj++zlWY6W43aCFkgLWu/TZyXoWrUDdd84w5LLIBXnb5KuyuL3r3lXZpr2ru7Gx/GKNNrJOCvqtpfkJI/YqBvkQ4+DoB0+S9RAY1hpZTQ5A808w8W23OAjzbbm9Ndv/tC2aQby1eQk2HFSRZDJzSE/YB8VtAWa4nsH7ZpvjBPCvwg7aU7m0yAIY4TkaaFA+VTyNYqGNkCujmHws/AHeTx7rdVRMCVqXzHTA61y/Jd6QI4DJLch6j16CZUoQo1yUTaNFvbmPcInB6NyUgBxAVwU4BIaKbXh/+Jjfz9Bp2+Z/iFDvsiWLjiy7guCQNdl9optNRvr/DniJKdubA0qzMq9uu7v5Ey1uUGSKt1+fnEwPmfEbmgYw0u223rZgkQOVA2sB6vRQ5UoTO5UF/pDoceb7GkgOfew2J/8TVr8sNm0BHm+9LjpyvptPOUr66DMZyhHS1+auASE1RSfAgfcj0BE46NVRtvwxW6A33oK13v4l2ra5dorlykRAHOsWu+/ATfcG9wa9diakOdIGoOue5p5BJxn+WhQtL2rWFKCQzGXRdrhBsJdC8AcIdpkDMJtzswX6REiZTf4jIJB/fSryz/G1YDpUgyaj9QohrMFEIicvWlmMtK89AfhPx8cdC3XcxtzBf9KEQQbc1UsxtZWMQ4D1y3XHCwxr+BOYREd/1U8zhc9t9bGqP1cwvpbk8xjNAtEc2oTdsIlqDu6IgDCz1rjJcAV/QH4jZ3O8ij+InlSciHuBZQaumEUPAs4u1tBBeG6bKjQyiTInCjDG53mcrWClBRwLSZIYUGGCdgQtCSy1ItzEFiHFM3OKWCmRbaM5yHxBQhQ/olKJ8WEsiHZSqrfEmmBg3ssVoyWKJ8OdC6qlzoF/HjUAbmsMoItKaNmRGGUYv6h7Rf2/pn9LfgToADqy+f5FFuR5crOEclWcZ0pqBs2SX9gbSjqAIl5uWSiAVZYaj0XbbnP5nh+up5HSw0oTau2t9KFr1KhpXFI0lM22wEyYvpUxBpOxnCiqcwNIUEaHUUP6T1SJXUCNE9dlYrxloEbRA4JJo+dryMz9Hsy18rrwX583ZGlV5UFIi82RkvOT5M9TQCL8PZkCncyfJieiLboZ/a2+sVo9MVt9JMPVa/P0hJtwRfJR+sGfhMG7jkwfgCfJcwJw9NSdlu/u9MPAQXQ+ZrS31jfRPzF2INo16bE8DfqQzi7VhEhu/RlUV1neZIuPrSaPlxlEn0jJ4Q/TcLXs1qDlSUIagzCLuizZnDHoDuTBPZvZ9Itqg487AEv5sGAPY9x5uWeYqvaXRbld5gOQUaxIpMpAk8K5CBX0YZ1Q/oSXaqIQBIH3FF0X8PcgWgNKadn2LvjTE73kv1D9ufxtaD9TDhnKL/+yYYtzboQJPP9/FiGjOuUUCMCSbwIlH6Hk6YlTwJEBbOKWeCGCWYRMZFcKxAqwmgFrp36B6t9mJc6KZSsEbWEBG0LIlLzkSQQOPf2CBVV42JmqUQD53VwV79F+VimNea9Y1Zoy9lsba8CikpDO8o+wTGKN0MqCIgDDRS1LJ9TtPj5R6tggDtDRp8wYjIFL3kygBx0HuQQn+Z528KQsOpKhHPRF2sUd2IxH0maLVllRwpQ5CDJCCgffGiJ4n8dEy4DA2cgYktzESrCG3MdKoMkkAeeCQ0XjiYAEbLSGiKkO/I9xr2tGkXV0ku5xI8HV5tQtBhlE1ADqioh7sv64LJqxCL/PUZRMgVrAyEvrj/SodSFsfjb3QJLU/HPR3aXtFsjiC40g4d/RH6M46dab2GmW8IDRUdVUhsNPltv1ph2LgUPfVYt7k1m7KIo5icEp0DfIVpJRaGmn4CrzSCeac4BCajRt5vG2Wx19E08dPc6DAIopwTukAUuEplZIiXTMmPxw0jTBBYvjBP3N8eIuw/UGRxZZUT8BPxKEBLRR3sCiAy+qUljTrKy26/FEjmCVY/Qeg7wyQJKquA4j6yueDkfRZpZNxiUcMkwNUuCZ94nuZyTCaHK94SZF/S+BnKNFWbcgZrcV7DfQfnPxKY8gEKxjfyKE1lIkgMMDQoXoaKB4ASOwd3eTS/6raWMo3KyX/Q7cd9Dc84f4A0QSjk5BKHUgVOOXRXVWtIAS8HCz8li6dLRXnx8p7B5haGxn0gqStTs2eMSg9lg8TyVa5w56JxEkCdyBZCoNL0MQNsyYqB1/j7mO2i9DHwW8FxCsWUXEKViB4ocoX8IVeCMPLZN4MiW/bKK79R0bHbe1l00JMkxxAHevqEgdC6Em47IqgA8TubKjd4ml5PNxU39m3QQ/kBXEcK94qNfsi6j0CQJosLEREaT9XNzvlUNFPSfHYeq53B4JDB7cQGaWRDsoWreC7ghxvQJjlU0MYJYlSvCurdjl4+aicRCcDIexsfcTa+hDmycJen0YqdeTCUW2J8bYBQGq8KuOBRYZhOZbYBt7jewtGaWnEVfAeqpffJ761XhWZkV+E6i6hsmQMajGDc1ErMWrzVuDA9tKR9GBaLHMB3tYO+vpgPW8dhjSNC+c+OrrmmVqH5+2sH/RQgSO5Sts6lUgjtecjyLp0jWgVjHbSSx0TU/nwVjdnRlUtiL2OuomyWMagfadl9n6ZplFOp6v0AobMZOrExH3F4oEoPTvaej8BrU5MT9M9ls7F3NlYRlBbiEN5p4o0HUsjp6H2Nzs0NvqmAv0DFGeMSJvJ2Qu35g6yDfxDPU+VWgV80H7SfhVP1JwbGKaKrKyMDWUpzVkbbhWxF7rg9xLiN7cwRIaRsbTPx1qZAgZRHZk0VJ6yNhTZi8LJ0FImHGHkpqjpxizhp6GvjFpj21U1540d1sw6GjDBI3zu4DGGVLHcph6C4/pFcTuagWUJTKZtg0a5gJ07PdvkaoxCDtm8rhxyLYHDEVJp7kekkfkox7NBqM9eeTQQL7KkC5GaYUZKgHHlkn1W+jJRw6vxKiTkhMBwRtP7E1XwJvy/hDL2t9T6LTEnBAQktcwotnvPYLKU40nSQcpnqU0VcEnOY6ewdafRJgG+R/RCdqc+sVfHf4eHRY+WIUQBKmqsPhgKI0fNMDk6WoX0c/JX2DEK7BI7yDC7m51hiUaJammoM3K+nabS88NnF6SO7YbrISNYUxTA2Frz9xgARX2N0L/0LT0mza3DP2WK1rbFZYdjw28vo3+ucYoTH+XkEws5x2tt0BnNzlwxHewCxbVN7QFIj1MSi7krKEFBOPbMSSWpVbcQCaD6Zd6Q07tZFwrl/AFSErYF+VMagJJnhPoKeoLt20DqdKcJNtqd1AlPs+trMixNRyx+jBbp5acxMSCld7ckwcQPcDyoDNyFYvtlvh6MiOnxHRORKud7ORxzaHBTtIH7N2M7UFMUMSZr8mbwxeqI1UF3gxKm7MtREYXiEQxHOye8c8JfIZyALEWGAhtT6p+h/oCyjpWS4PdgdCH7XrgWIjVtWtMNpR+Ia39zKUT9BfYBkRcGQMW5pIzuplhIBEx2OVXAoA2pqnXJNuAqey4AnrXzbPlBZS5+Os7HLzdNn8I+hZqi2uM7cn3JksSuGEyCbYww7Pjx+3a+QB3/nDVdt7cGlWsMy5TVQU3D/KGcwpgvAFgciNw7ghURTFqFUTcQVTwoE3CYpuWDMTRq9Nvz1+lLy/OX51hqt+jEcMCjdNoUlqxFNNl2vFvvGMqDFcnI4FjwTgfSncBzJYo/yDxBHizFPk/QLwUs+YjFZC4jVkF6IhJ03OpJSCN3c7jMDhHko+JNM0spFxQKNKkr7Dytalq4n9XsYpFUs7Mvxv9Ym8+xCuaDDcWyS8c+ION8BuQ/7BRSZsTZkIETUcy5irmcqhHUHaxRRFU115PJJIx/jfTjGxHYHU4zcgKxRYooGdEZYnH4JzphdGDAN1QE65gUHLss46ArgrMypJ7FN+o2rJEJnMaCNAiKsYo5RaFO+T4tpquYEm2OGCmtnvIVlpH8hCQoENW95CwlMBqa4CYjSYymPYnpQF2UWNTzBYyn1Q6V+xikJPQMCCPW5F0kAYighARIBOv+MJZcKJfhSg7XSuWxQozlL6mLJg6lGeGxjoNchZVlDlmwPsVSWSUNWaA8hPIkggCyzC2sisAVzoBhA2slrhUlXKLf1fmSvzx1+9yqIQyy4OXQxnLAO7MTjWbRiFG0ptABg8AN+1hDCO1srV5T6dXHsIfUMlAh8MhQawEd3x2SkJiIg4ZYmLWgTwfNLqhVsLDpwwhSvi0BSCRXLXNh5pxwqjdTlYjGqZAfWWiduZp2v2brsJ7UrlGLcFF+iZecNUN9gMiVo3id83O30adHDZZcxZiua8MLFw/dv/Yn0aQvR+MPnaa2f8S2g9e1mKAxPXM8rlk+YPYXA/O5/dBREoeXGdwImYB+a2QNf8V7EVvipTSd8nRDIXJiPXAA2X0qZOA6sAabi9lhTyThhHg0wuOr/DpuAsqIBzjHhO8pcNxGzDJ19kMD6aSgQ4OvzjkhWfHjlrY8y9g234oBh1foDdalnIINzlwFZ+BRbGJWoCcjHjCI4WuTRXGQ9XrLlAxF3+1CJA5YDLEKp91DbmpJir0ZJ/o+nA2kClBiN97se+IbhJQgtiTF1kv4DjWm+gGzvEWcHKwit6d/V3ERtrEgHiqY19iD58iFtsGsmHBMUEXt2DTj0eJxzXkwBMruVoAn59MLemtt52Af0t7c5foiUIX++KtbAkgLJgak51yxgyBL8E0jcxa0mcfhHmraBLSpBtKOs0pmOfaY8q3NKDpFF2z7R/NJDtiWNSH/wTWRhvhU4FMr7Yc4bGk4yWIUrRkjnnZIplKkXgbGi94ABEyu4g4wNZ6dSuplXHwoMe5c+NIdO4kvamX94GTHA/msR9eKtBZTmoRCBl6fRUKtl47WT9OThqqSW7lFHjJQm56mm7plzmNnXQ13dIpcJrh2SxVFR6MxKRp6AyWhxiR0GaO1Ml1c3oUOW3BBlwUHMPvo39AXhKcP5JpSZFMSzpGkS4PykUijSyJ4Fx/a26tGZA0EsG6xehK1SkpQscAiKxAIKJfh+IZ9G90g3cLwPt1MvLSxsRM6NE7VqaK6XH4IJmBFLfMNIM+foaz+u0wLdPWgtF5YCcsXkNiB4R4kM6czd64qUtco5gFD/rLgt1xQ9YURk6YQrdEGWG3M6WG0ey65wifxqt+6R3rs5C/GTrCN9MbJXZJ+Fyhxpx65x7zU1XwyT7Yl2drKGaDtv8woMdNeCwdcYcWcyrSMRz+MYtmA7vysTgX7Rwvj4Mb9JjjDW0oFyxQLnuzvA1BUsZ4xu7eeViaaGmMOQHqIeAus4JMQNrDf+y64lnrZEGx6PFXXzEjKFWLPkZab/gKD3djQCm+YiVCHKDotXNkdKuOCEJNSJGhLEwR/4/1Dmzcs3nQt0nbv6kwXtK1EhgRmrJFCX7slDdVp9FNXZeToY0KuUsiR6nQAc4IBijopYGNqY5seoh51DaJColaQR6Ztyrac8gUUarGtQJBDfGVBjei2To7ZBOHAnG4DaCS2o/b4uf8GK8pUYYBjAGCVqgIhNmntEskBI8OxO2lgwchRDrFysBVkNSHwgEzV/l+BnjA1juRRtGAboJhLa2jB3qxBneWdCIR3pxguM0hyp4c5D2PvLDsgysjcXpIZGM9eDMsJmeMWxbhGgoP4saYhF8YtRPaXM/Hcmt9ktzlX5bFLeb/KQdfUIcSkhnEm0R+gg54KFtZ2NDmuZp9RrTwDFJIxbzJG8qKF0ZxiN4Wd3XBsQ9J0Y2Ks8SiMBaUjS6nCOJyCbQSVSA1hpNBGRqtrBIwQgEbfXC9qdLNsgdhstuVjQFtvcoKjyRiw61YVFEFcyoPO2MqXns9xMHWPrnJXbVZ5Mdr7bWBKvaLkBIlSIFJFUs5GpOKSWX0tJCFfjNpRfXhLtDEcCICrVSpuVSUz9s4AtteP7OpxXtsixkiNdxvoJ4/dCgt2jtxbtOAo1bcruADqGD/41Oe7oUTrBfASGAihza1kWmSch+Qx6xFGIBJb8rqlXMQDG1YxD7+mItnriVDryc9DllInfp9GtdocK9BcHAaFfQn7ZuEwI7s/cPePhQYd10mQ9dfeAZoLCJCmBWgmWo4Yw892lSm4wRcW6VmRLTH1DK9KiWoOIZ1Tc73WZ1TNE/53ZYeU3EZGZ/F8AeFke5TfTUCauapSiG6J1MbL85ayqKhrBcVdvFMnBsKmgw4kPsk9uOkdkBy67kHhHSgUIriQJFBHYHSoIgM1HOlVqBKj3wL1JScHCjax7yOUHgUh4ba7me0X8pshzCcnUSt443KBkKLyKZLKA1agdo8F9pgdHCyB49KRJrtIptk530UPAnMAQebbOrN2BXpI6OeuUw4s8CqiZo9ZvpXX91kVhiBUAK7gQicmRPEMN6jmOB/fxpbsl3Mh5MJMQjjZ00rNlaZ+jFKR/siIxJoqQhQ4J8xvZkkKZ+nSL260oLl8XHtq9k3JyfXXlXuOEUfwdCYVDaVBwi41NeZDKGn6WAbueFgGZKypVEYsCbF0moHKrDYV8q/ug40lW5VsCEXhpopXyvYTpReh8xl2YBdZOJVPz4lpcwBlXs0lhhaoMTqyFJllKZhvfGNAHGhlBvR9MIsjpoHlxQvUqMbLVIXF+OQPiZ9iqrVzf/5fAf3nMA+Uaa3jJR+xQixSCeO9F6GsXdihR361bJgfn0Y/Bd6oSIhWlQYgN/ngaLf2uOFavm317Q1BZoR8hC1OOhBeVIUf5ARGoxHbvRhSb4nAsD91ptn/EKw1L5NNYhyslCUx+pn6hrZqyvbjrrm2V/bt30JS62DrM78yrsr1DHFZAwiuJNnak+6YMw1NgP1MHPCVm2czArCY/xkasa65a0BkMfyxMx1Rgnj5nvr0zQSePRg97I7fgjA3nmbfGMJaP4Qsjl2U7xiQZXB751/8onj2GZEWpxulhvND8M7zbvWh6mj//MHb0NgN3HC6aty297NbePKPI8u2N5YFy87wKkY2j62DBtigzn/8WP3A3aPxxlz6ykAC3hhvsjaPNSNpNO5/tmXJGrIJ5XD4wmq/lTlt57wFTd1S2noJ77IHW67M5BwiZBciagVGmOv3BLHdB2p9cumIEcJCoFUlTMJddAgtANjH2+ybEkj/uykkMmTCHqr30g+0YJEpJ6M7KRpKXKiucoO8SkBDwihPfVlIaw+b1kCNgJS/GEqe+KenjDNh8hNSfANfTj1EhJ6qIx6xjZAzjJ1ZB6MK4S8lHDa6iN9kgO5nNZqDgvhF0lszsM+1cBKzfdaeT1rM+819YJiwtO7Mq3MNpxU6M6tPhYL4gF2wynBiVspc5qZ/VtENN0baVP62gv5wYC5d0v4ePQ48hCXqnHS/lLesWbXkVw+vzIvMof9Ukp0MwznycRKqRQwZXMnr0XeuCa7ls92LbbWIBdjOY/Lch2HSvUBXfls1xKGGbhot+DQikupyUqbWzab3UqnOAwd/2Uc4s2K/ceIvStkflRm2q8ITPSv6W8VnegjzhXfJBgxZ6jN0gwHDKl0kEagZfou7r13prQunoHzUPIYg2O7/p/dPWMOVYHgkR50LwydNePoCrYZO0fj9DSDxrp5pJnm1I79jXO5OQbgvSNw+uQbLcvBJ9IAlnUkjZvrc2Y4/UcfNINTQPrGI8KJfZaDO5GHOeCee2Jly0VzjhzaZ/b8AmJs97X6LIJXgtDlAP1SWwn45f5JeL8DSv7yXg/sL5tZwfY9yAKx1rkfxzvU4hEe01Vl+JdJkkwPdRyDcMVU1I3RfKEeXRM9tW6KReO/qyFdUNWBs7l8iVNKidsp3f+b6pZ8wDNtyRnvnVzvJoW/DtDmoDsLBlZxOhqQrBM/JkDnDxX57b0TwNJBaObq73SMDTgTfVxSsnroQK5zFNc40+nSjv4GCJAAX9WR03Wxdj2tveQ9ug+m6uGbqzDMXogzerv+q3dFT0cM6ldesitvisYTnxdnPRfsSnxd0RC9sVIuhTkbKOBBXotri1DyYr6P6QL0fGplbEsXurXLfqUEOSduCOkts4r5hj3E+aFI3lbG9YXGJcJy1HiC34SrqEE3HJDkkCudoT2rJ0EJrqDrW2BvPmkGopoQr64PFLjHASszla4X5BEltANuqTIYztiz6ugzWNr4Q+VD9p9V0ehDmX7GFRbyVW8bbdAFb9/obddvUCIci5+1SbXv3mAH1xQKg4xSPPkSvEZDnKpgMmUrQ5489ywPX7lNHPrh2jYVGUAPICSGMEhOl54RgN7XID0xVElV8kKnBeEqNGaT8t05qEGHwBx+K4XHFHhpR8DAESbtQ6g370yBbWCkwqAbB7/5NA5ZIyRtQgWOzBERfW/9hV/banSqvkNmo2kxBsa+c+jFvvwm0O3h6H9JvalzOQeSkezHv6qK7gBBS6btXF8fZrLnIsyepZMHmPEmO320yXeAQus1DQGUlyVQXBaWyDxA5h6Zsq2UifyiScNagm56ge+XgV0GZwDFN7eq/DMFhHH1T/TRSDjC82Wq9E4OXiIdtKQDPe2Ugc6f2KLc1aaSIWeOS47h/1V89UCdQFgfXQuCO9ldR8QqkHPb2WpwF+85hbgHLLIs6vJeoNoxl6JlbNrQUgnQ88QmzIHI+YA/fnDE3FIqQ2GLnrC6F8iah2k+FLWQS3hlog3TeyS6NIlraetVprIhZnD0o6ux7PlcmV0B1V8TxeJBcn1JiCn7r0MT2zcI23hzBqCwgr33bR8HRqU0VHBIv/e21fi6PnETs3gJX1FSO2XMt2u4xpQ6Q/FnQAPOLu/VQf0lkYd5/J4+SiCj5aGNZgVLywg4oYdsRui9cpO2rpVY5XCeda4ImlFamPy6BCe+i88tyAf6TAk87B5rSKour5SlJl9d46CfGM6rfbrJdmgpf8B8YabjbptN3dKJGPa+wCgVX59bQgoN3PO6pnQM/Wm42E8bHsidYaB0IY/6hId/sAetAcy/ohuS/GQKs466O0n7KT0NDNcbahhPrh18pe7LmKqbMaahCy+mwYst3PONXrAAz/G478Jnfx6VxCEOGKrjzT5aN/cQyaz4SFrirT1XEZ+hhUryZyJ/uAeW4g0YhuJAnH+yik/N+J/BlR0bR2H6LwXiQ9eIaPmxWfoNa561/BYu3Kg/O3qz5ySVuAbKQ4sMLYjUK3Vr1tSvpvKJzCBGXwMrUCGA28GLvgbBbvY0VYEU1ZMOrYQrq14G1G3cEzOSaWrh0sl+IMEpHgjOu0DRuC1XN/XvWTT5xyM8g6FGPTfc7qwjjU4cSV+9azmS1kk+8R4tXnFEiG03o5KUYmYlpVKt/kV4Rt48lkolSOnHpqc4n1u+inkAAXSelTGBPSpFaA69hDsFabKU+oEaBIXh2LW20StlU3waPTOHa3y3acsiWmkzo5b8sKPHoqZsDxziNCjSiVGFzmeGKG+w2S4gCZTxtndEgeDEoaPa29QdmeNw7R1b0NMOjM4gxpD3NjSf8PUt5iyDgwi22hcCCDayz5ANXV4d5vRNDds99z5N9thD9JnlcM743RZ4Nr2ty2WPiQL+3adscY92GdxLkvdVuwHT4W6dNR/NNNKgyYMHp0v4VHFZdyldp9AL0/jOrpOiHpiIkhnqcoY90FUD536F/mY7+4Cz2lt65Nao2XLPFza46v/KBzbUJiD2YN3Ligey+aMn8IteJa/B+oGrdhZqPxhe4t6SqnDa3G6R6t5Sydj8JGWaLusFfDjcaJlkyyV2Q03G8dERCBUwnTAnfc5b0/IDRcZntgcBSENkGIr8qvYeUGh57AOE39EeBMN646ip6z2wjO9lDwKUpxXd9vTNtsGWePjgCO7tCTSWn3MbbM8f4CDfw20vv/Q2PPLsy5ES/wIXdD5efchqsDlaM9x7xgmKMYf6aDhtN4+VuWNRtoBm0rS6Cgq+A+NEFbECpSoYtY0dYSiq20R4MnyZpnHh/enbi/Tv5/+0r7KWe8mDNzn5QDD6oCTdxPxQDJIS81ybGK+MLW38Wrz56b9RILGGmvMnCPyLnqjQv+ZJ5fxQuZ/xY+bfiQ6MN6bU6bGmEb3GJI/5TFLAfta28P4P8AzEAY27+eeDO9ATO37hZEmYKRI8b3gyv1NgeEdcLl9ZHzOQPpGsYocibTNw7uDJ3xIlHJVGCMS32FwgVg2vfci8cSE4dTwYPUie793+N5Sl2yfTIS4JV2LimO5b+97rzjz3iddDPpkEwUHvX6uoUR6BsJDHr+i6wjRF6ZSm4sZCFlWj/wGCFvJCI4kAAA==", "sha256": "b56bc8a1cb8fbcb32638b3f74c062737db9e8b1290bb00011fe61dc19c868888"}, "run_cfpb_v052_blind_semantic_judge_v02.py": {"payload": "H4sIAAAAAAAC/9UbXXPctvH9fgXK9oGXnGjZbjqda9hWsc4ZpY6tke12WkXF8EjciTaPZAlS8lXRf+/u4oMASZ3sTF7qmcRHYHexWOw34CAILrqSvXh5/h27Of4mesbWRV5mTIpdUrZ5yj502VbA1DN2m7fXTNzkmShTwRqxEQ39OjuV0Wz27lqwXZWJgpXiRjQw3+5rIS1CxBCi6coSJutGSFG2kuW7Xdcm60IsWNnt1kARlq66JhWzrswBIAFeALgqboAW0OyaEkBgSbZO0o+srZj4lKStRmKt+NSytdhUjWC1aGQu27zcsvZazGQLtJImY2/11n6Ane2AC5ZWZdsAkWgWBMFstmmqHeN808FignPgsa6aFjgpqzZp86qUs5kZa7Z10khhvj/IqjS/K6ko1Ul7XeRrQ+YcPtUEyAd50+Mn5X7BXuWtaJJC81DvM3UGGuRlLopswTb4F79JijxL2qpZKLH3A7NZ2+yXMwZ/NCJInaebes3hhJ9xOmBuDpjTAcPMU5ZIOOenhEjr9yh1XgvAElzuqo8I/Iyn1W5XlWaFkLDwz0qf99s6KRd29O3qx5PX785e8IvVyds3r/mLN6ert8704EicmbbJ0/ZH3GE/KK+TZ9/8gW9y0BsanM/Ep1TULQPArhCvq/Zl1ZXZqmmqZtlvSKZNXrcy2gpQQjrKqBS7ioPYEp4JmW9ROUdb+iLxGYa+bNHo/0nWs9n5xZsfVi/e8Ys3b96xGLcduUOzi9XfT16dnZ68O4P1HaDR+Ox09fLk/StAO/mHhnFG7Oz5xer85GJ1OgAxww7cmx/PcTElI4/NJywAQ9/kW/mkP4ondHzySS/+0cE+i3ZZMJvbNd68f3f+3m5dLTTeF47Cipp8VYuyqTowbv6fW1E+59/w59+sefJ8TeebtK3Y1e3x0ycpGPFascafHbvL/vD+9PsVPzu1awYHiGZCgG/ZNAn/b9bwEvzbdV5+BG+jtddbFFeZzdIikZJdkH9+QXIK6cScgbkyJSWYPFsy2TbAzpBDS8yo53vw5KGjXJoORBCHCjm3sEaumjJugn+Hf1m+4pcnR//iV1///P1P2d3z+59P6a/57wJlX8rpL43bvAyKZC2KYMGCbYP2DxumD3XcIguuCK2oUpKwv/QuL3khym17HT9V5DGYPAYDUaUBoBzCyM/sdVUKC5uJTdIVbYyDC7YV8bHCEGX22fCE8NeBfw/xOw6SDWxaCwJwmZ4XvNpspGhlKEWxmbOjP7PAPYhgaU073zACimgTLJfE0Jz9JtbjwKod7dHo7JJcCvb3pOgEOdnQrnGEcVuJ5Qni7zqJEZmtK8gedNxnVcOSNf7S/GtufGYg3CoJJZSP9PxMT3wbO/hfxq3L57YRIMQGUoakVNtweFQJCK1jtRz9qLgQsobEQEyoeSbSXJK2WTVNUgxWqJqN+CBS/esmF7daQ4EHyCTkkhWQwFyCBl4N1YRvIGGpmn2MEFqx9J44GNYkqt3GFI2FM6tCFvAch54cA8y70qrOIQWjtKvY6xjX1XVBo4YHyt4i9h7kvoOl8rqgRJEFPkFI1IDgDpKoHDf8pz55RDnbZBMtMepR5zYSkX0MEqIw0OLTB/dXOqedaK+rbGwtGjZMC7mYkDsZkP3q1aorP5bVbQmylRCTRQYG04YaHXCmg7Cn7JrCI5q6Cd7rlWxGrlYBscFBLdmdpnM/oaZjzh4Umac8hwSXiayDs06V7DZWcJvPkZpmDGfCDOwkQu35KPYyRALzX+jwjIVxk8g7rs+zTt/3kbcwuCyOmbHL3q8YfXjEmyg82N1/uhw8HOAzDK17o0+04QlXd3hx355/KQselc9iRPskYgRd7ZdIQuE6bLSsAEzw+KXQ0vhFy3+RLA4y4RawD3p2VC/IBCk/5jaXCG1CMKXfUDu+VRhsVW5h8lpVo1Q2n+/ffnfKbq8hi1ZBsLnB2s+rXSVk8TKiEpSyD1PEOYVcvZdrZYq63jmjcdo31h8wumTst7BEst0lS5AfuAl0qEcQ427ypiqRvyNZg8A3eTrzZQi5XpvvjBTPSoh+RaHWjOPj6Hn0e1NaYxmP/NvmQDBXgQAYUGmRkoQE/0j4kRaNaMIiKbddAvlNIEoIfCkcTRm/TAop5pFGI0HPZ865XOoZiO4QmkJYDTasxyCZ6tcjlfJAr8x53ibFR25zQY7T5VaGN6g5S1V/Y61uMr4+beRZnhTVthOBDumwCOgrygcUSRFYkFp4/kUR9nR1j26XVlmo6YH+PUydwnxPDLcPibz4tGCQVOxQBAIaKMSvQplPraxaGw8IAglBdyG4Q/7uL++I/v3VyEgeZhJd+oBJcO50PioUTXEGlBCITL8qXB9t/qBbz0tHWJ+5IVrtEohf9duK7uAbY6VSinWXQxz0s5YwTSQoBFY/FD1ewGdv8G4urS2fsJYT06BFlyqhQ1GUyU6gLJDwq5PvVq/4y7PVq9O3blKBSVNSQ2GX+ZmXV0qNBKQqqXgTvOJ3uEoE6RhY2vw+WIxglbOJTak0mjfFEVAjEBkRySlSaKUApuYhE0G5RQrpEseuhki9JulIn3YNaAaI6Xg20GrHtnvFnnDJtKb9nC9Uyg7VWS9VVVDEzAeNNrBSqAkuNCd+aCK8b9nxY/FGOXfLsSpQMACTd7crQpa8VqrpmBSWHrFe6msIU6Vhaf7racX3ypSXx8+zQxrh1MwPa0UPxTEENVjGTCuGEe14PToh+v94EvYYw38P6411CkZ1AHo21J7Q8LsgXua+GlkaDzkOpSlmuFcqVXX8ivZ6+pknYxsYB07Gbnn6OPB/h6WaZxiuL3FTkeKQREoVMsiPNntlnD9qKiBQs0BpbUvfjnTGhnJqKofpa4TAC/m0nnbTAJFBmwvP5UHvDIfnuGPYyYO+XS1TJ/uiStD67izLAU7DxoOl8hZSgFbkjjYGPjmAU/JSFUvW7WpdrHygTHdCgIrWvbvTXkGgLwHiQgFBw04yCLBbSBiRFcDOqCsBeeJeYshvujW0GiJ2oYj8AFUmJLjFPnJq7OBF7gobp9l6jyX8Hh2SPmWKk1IncH4B/1P5U9mT+5ouOCLcpgy1/BZgWBIvShKZ5rlK5BZkh2UbP1tQ5OdY4sXvmk6L3kRfW8FBrs/1LY89sFCLSJVuS7/PoqQ4oQqL2ThUY3tch+r1HjccY9VstXypTsg9qodV5352oPpXvPnFCjYDcJZWns9nD1X/Byr/6Tu3celvcuXJ+wHfC2nPQkxdKjFcRWrQdxLkO3ywsSdR7nFAbOzf0bf7QJ6nn3uJo9ZNOIxpuRKwze3rpsJJdBFFobb6ldYR6EuVrcrvZzqbhF720mt1qxllWEgLKmcqAw6omSLVNFBp2lay6uhqFYQFtT+SyBfksJgdxCNfZlXCodZHGBf365gN+nFonv+EU8Oa8iavOmmFxeCzwJBBLgOdw5Nee+yF56AdtwleKB6gqKC6FW4XWAqN0K2+RNWiuOtZvQ8G5wfKAdcO6yrbL6kewAp5gcK48j2tOTFwoXf+pqpGDV+qk4rs2cqi2175KhXAcVe30MgsCrwPRoessQYTAzTdG+DQeIREFcpSB3M8N0Cmm7u0Kgolgh5zMDFAg6uYHhQ++ul7J8CoJglmYSPBiBJvyjOXUw3L9dRgRcjQii4TkwhqapqHtqr5xx6LPp1pvAWp+2n69HZQizanKyxIjJKi3bsMDOcm4qEy2QgUD/4HtlgIunWPUmrN9yZAQTc2XFAExrKjZ2UnpIROg4wvPbnc6UKTBcrisf2ubQwGPS9wv3gAEw3Tx3NN1UFzFBbv2zCPg2hpuHaGHDiQdx07wnekq25QwAdq8cXWSPxxRwjJJ97CTXIprajsiHPNC5mOmcffzkwLYt/pwO60ApSf4eCrd0kbD1QV336gmDBf4DK9ht71IHf1poaqTgB4lkiD7ma9a1mqAbHrNM51A0kXMICH6cbUvFnRyye0+jg8hXMf+X7SVHpvF/c/F16So65LIS782mFJDXB1Qf854Qr2tkmgA5mZu9/YCVct5ObicvQ4YODDdRo1HfjAweNf6o4V+q1c0PsLtqJeJfZ3x3AY6jU3XL0AopAPIUeETxfM0VgNJSELdav7Gkq2QYm1CUg4FDQMcXbnr3L/5G6C9v0oIIaGUHw3FY7uFww8uZ2D3zAypKFU0lUuyoUXfm9ejw2RsSMLYdgugdFJgGz1OBgGTGTyHi5hHEu/G5n6/XxgLZuik9exbyXzkX27ZzXqSg8Ap7Iwr49HOh+rvxZTbT5gWTuhiWLYNYPY+5qgBUofp4kUU8sY3Y37nw9Vxjo1w4aOTUPT6ypPhRz3KSc6Q0YLrZSuE0ldeUUj8Fdyk0R/scvjq0hHs0hDTfHotGQ1lOpL44U7zutB0xr/ZVswXEKBSldMg03QgzissaYcrK340BgMjz6+rXbiR+pDtdCC9Um0czHS+xx9LeKt0XsnbCB9SmfTCkKduSYEgPnl8o/Hx1c+o0W+zSHr0o+SOBwtPrUhmXAbH/UebXXobxbdAb5YQ3kN3OC3k84Pb8XMwr45g2eHLMIwoz950mVTHSjd2Iinuxzmj89R7H9+sQGTvCFMpouJhpgSTtx49f3QNrBmAcmCD+JGBrH5MUUUAa2EY/vrEWMfhUn3zcnYZEbgoZb9iLBl4DOvOJQx2kB+osgru+yV1Ll5s9gmSQJtGEb0QR/gUT3QF7+xsjZzD7yYNNnY/PCnzfWuoWG+fSiVJ8EjtQwuJ4pdMDWb2xzVfPtQOiWqGr7tgBP1jJLyo9jLlnysvnQ4VEwoYeAjHZBS16YxHgz8DTZ/6yaLozvlDzaXIsu0nmHCNKfV//FDOmB5n2nDuoN0QOM26i0FU8pu2wMJMg0vfCCF771qf8GGb2fprMKvvvp4C++lJWWbfla5dKs/XL3Hsgwa9MXMVwmwSb4pY/vhlih4N85v8Bk43lz4BYPNvUAx2gqK9hjyNCyiue2TgJglvux1sOjdjOT4dC2+tN3ixagvfOXXAKT4HNkPafvm3Xj0GlQM7v1T7VdoEGOSBThpth0qzznNhO7bLM6zKuV87mBGSZbhMoQSBkdHTXIL3GE1Fp/TrbN5Zei8sj1IAOpKmMGbh0NUzEPcR0ihBT5GCB/wHiQDSld37VFTVY/Qch7qHiSIFnVEp5iotDBQNzoBkpQt6IUyOem3WTU192j1ae+SvFTn3EcMBMDHEC40jdc5p+tv+PcCkX6jAVdPwNib89XrC9jD6oKfnJ/xv63+GdhWIV0wKszlwUccYyIYzHSHK9MEldsAFrzXv1pmYA0QYa9bpZQyUuLnKH584Izvi0FWdQVVGLfAEV1/EAE0541o02s+DRr6votp+nhZH1sgI3h6BBOzCfcAiq5wiEn48lompMLOvBlygSg8uCB+ZWG8uHRgBrIY9SgkycG9ZUc2jZM/QMkDGxHx040DZAaAI0Lj0xjIfeRMH1xK+V88GAWpNKDHfyBCaSWO9d9uPwqbqHgnq41PrWq+3H9loToAzs2U4uALL6bQeMGyOEV9+Fc/+B6FczRlzvWbFGXXs/8BP114EzM1AAA=", "sha256": "046cd0300d4aa751f3d12d02e194102d3cbccfdeb19546444aa7e8aafe65e0e4"}, "validate_cfpb_v052_pipeline_smoke_v02.py": {"payload": "H4sIAAAAAAAC/+1cW3fbRpJ+56/AYM5MAJukLk6yGe5oHUWWs8raslaSM3tC8eBAYJNCDAIcXCQrov77ftU3dAMgJCfztGf1IAmN6urq6urqujVc1/2wWCRxypzbMInnYZnlzu3uvrPA36O3Zz84F4zN0fLNeN9Zx2tGoKNilX1izjwOk2xZsWI8GJyHd846z27jOcudrCrXVenEhROvVlUZXids7DiXN2ioB7nL45IVTpk5YeqwzyXL0zAZ5ExCxFmKAXIWAfZ+6FQFYM/u52FaxhEnbslSlocliCuiG7YKiyEQzZ0yZ2FZDAq0cNBfq/lyxdKycMICw6yTOIpLhxGhacQcYLgBxeUNEZFG2TxOl3hiKydOnes8C+cgaVklYU6dc1YUIKzAbI5vWX6vZppjtDgtwIH4NozuR1WKt/EiBnGgaXANzDerMP80AvOSeBkTQwau6w4GizxbOUGwqMoqZ0EAhq2zvESvNCs5E4rBQLXly3WYF0w934TFTRJfq8dfiyxV/2eF+i9nYogoSxLwkhCqMY6yKgXXxXtwnJXxiqmX6nno0O/fslTiWYNfGFSBneFRvCjv18Q62X6YYs3eYYFzrKnsqRZPgvwQFux9NmfJEJSki3j5Jo7KofM2Zsl86PyspeA4z7N86CyoPdDiMxiU+f1k4OCHYy+iPF6XxVhKBfqNU7bKAgCHwZwV8RLt42ixvg4gy/uBEuWAizLa9oMoW60gdJI8j+Omnw95GCXsPLsb6qYLKV0/SeGq39AqBEW4YHVTAiEKMo4kiIrbxgtIeJbPi0arEt9Ai28NUNyE+998GyzixBiE76aARk9Eo/+lvIFwQ7xAIvGowKbfxqg2h94cvz38+O4yeH9yGvx4/uHj6ZuT0x+Dyw//dXx6UZO4zCFwtL+CBbYoxL1QlLLPEVuXDqShSthpVr4lQL7uE8f5s/OGqwHsPxZVXC8IgSN1Iqb2VVGrinE97y9ZbRoHIswmDriR5WyaZqOczdli9n9HDr58hZ/BlT+69oNBcHb+4afjo8vg/MOHy+D49GfnAApszNLbOIeoLlnpuW9PgPfi7OPl8fHPh++sHq4/MB/RWQgmqSavhdsfY+wsuWWeYEy8cFow/AVLCiZxcBYHgdF1THzEYkxfzQb+4OI9ZqsGt2jZcVxxQhQ79ebbEXzGCt3vbFuKDOdHjiPKHZx/PFWojXGAOK/SYH93/9vdf9vfv9x79c2r3W9/ATjx5+TN4eXJh1ObH43exjlb7Gi1Squ/U2RVHrGgewB/oFb8/PAfQK7pI6Th3Y6lVPhTwcod8OufFStHxMli5zoso5tgl37G8o2r0Z6cYpkbiKXkzoM45czk6murBO/vjvkGaOAM3h+enrw9vngKeQMbbw2wDeMFK0qOuUZMQvnh4oS4XQueIQG8AWMoPkQVN1l2jPPdloEgrOYxuCH71Wuw9+py7+tvvt5/9cuOtDKCf4Y7iypJBA+wcLqXnoGCnMfFOitiGg+Ae3IS9VJKmTl+Q6xpSRAwSgEB5bXR1+Dx+THNejuKnP0KDd2P4eeT439s738bs7u+3mcfzi+fmAAxAMsN5UYMk3wY/Bm2KcN2h3FDZihMxRyGEIyjlDqESXLvpCGOoztnzgCwitO44JYMQJZQuPcOLNDoE5mFp8fnQEeGqLZAIywy4GED3jvXLMnIwMycMyiTeB5nO8U6PIIhmTuhE4HIa2HTkr6HjXh2chKcHV5eHp+fXmBqD3yFXbI2E3cC426MY2yNPeXl7tX19HD0y+7ob+PgLy9Hs5ffq0f8fzWmh9nD/vDx6todUscTX2hgd30D485GptV27nqv//6nq7nvvZ5cvXy9Nx2Nr4rZa/81PXuvr+YPrx6v/NeqmT/LB/z/9aP3mjoLuVTjFUXaJF0OwruP8Huf/9bdVc8q75j0TVmui9eTnZ2ri5ebq+u7u7urMf5tTJK4jl2wjEtSa9vH/9vQGPRxcIFDDJv7Z+jN4//+KFRHk09EBNghnZ9NwdL5BkdzzjYlSxJnxTaMzOxNREZuvvKvrscPu8NX3zy6VvewKO5gAGxgRW+i21vgga7gogUbeYMlmo6cGbfQeQP2AABp+zthFJElv1H4eGMUAleRRdgtjkK1Aet9Wn8BCPYMoQPenpxfXAZnx+cX2C6HR1KVbZnlyeaOkTB8lSQb5y6m31GYbuCL3LKNE66cZcYdqMy/Kl4oggBPen9TVNeruNxka5ZuYnhI2ENLCPsGdliMs2ezyBn7jW2ukyz6tAHaiCUb7HoGnwd/FzAg9Bwj6GugYgW2DKGQJzMxuQyjsj3L4/85e3dydAI1cfLjf15eBEfvDk/e2/OkKd5nlQNy+XRyVsKp88kbdPJ4eVNiWvXuGbw5OSctD23//uTieDvLJNc2QHNvsc7fYLSchpOTE5PyYdhlVTKXgvJ1Q1AsWIM9av6rdQIdtYG6io3nuWQgUz1ZzUuJY66QzNvcO/zh4sM7GF/ds5WULaswh8ZjbIO9sIpBFQxFqEpsg/tNxHLiZnLvGywcXFyefzy6/HgOk+78+BACSEoOtupvLMVpKdj4oLWRi9UlUoUaF14/jmeu2t3hFrCEpcvyJgA9K7I6TLgVeZ+BjiMESQj1HHBH0wSr0k9pdpfCdCcsBRmx/LTWgz4OyIyNEmxh5z0iBOGSedq79YWTKsbiOmCJOdYer4fQRx4euAhpXAOjsErzLIHNLf3nqYvgRw6muRgABwl47Ar7m6QdymXiFGUOpNx19nA6yUkf7NV0vZHH5h8kzGTtxElAzVTOWFBU3KeQc3JYIowTFNUKYY/7PgKplzJTYBOxQqJFj9mAv/2+4fp71gJLwr7n01yx8iab8wbIngo2scDs4EUJQkUrQXXRmITvjP7DbploQYCrALI91dN3QC5FiR6+HjrfDp3vHmtQvoYhtgDFMSrGXVmbbGdVFSVfQOwKR2CADfCdJkzOiztveBFj737WZNOwLK1WXHBrimwCEK/iBhcYLySIZsDxOH9x9p2DA2dXeDmGXFkIAC5Rj0kgnT8daJz2SM+YLiEoxKTDhMf6Sia8Uk5aPVlo3SpPNRsgwLSSPPAVQCaqiLzHuXdLA00ozsSXbA6BJZEZUotcMw6CyWtXXHTSfl+MrUyzjuSLIcdiMFGSwl9u74RRjT46KqV+OOVzScaYnPrCpEMsFA9/cICfoATfMDriRfzDFilB0MNjk0Q5iE2faJSzEiste9Z4BHfJ8AjSZR6uCq/Etuf7dehgh8PCBenfcR5DI0/LCseJYPR4PJ5JTlP/QpwJUPhzGMw4E6bh6DeYn1/NyBYjrJj8Hcs937doqDlHqD2OaiqkdCKl9aWTzjp2A20CHDhQtqvws7c75LuT9we5Topue3IsNU+tD2DKyzM6u5s0hEfodK7hr++h5M33NqQGbcRoJq24kLNxTmHCgUX0Bycq8VPwUiu7JnYuWZLBCBWfC34hug0xgQLTkfURIu446ueYS1XyP3ciEs6fxzzKLBjOexrqFeRMZ/VsY5JStHvAIKIu+pgDAZKV1ERgmj0cUgLqrcWhEJ6j2Rr7SRB4wCdKo5i7nhNHFPWdt7Om4Gt2CNzD2s2UsyZpQey7WqUkLh6cfQQZo5KmBHM0MB4xVMVUu3jwLeVfoyFPkLt4BrPEW8Ep0pT0jshuvWxuaT6BcbiGVTz3FoLn5EywlHZxEN2QiM8nDwLHI1SlWAitC8G0lnrUS6jcZbV+wHwTX8fa3Cm0Vyn8JG4CGVwRDQ1WiUbNsBpGNtX4vojdypqS/G6T+lfSQV49TUtXW3zste0EJyxNrdgEbihLaSxQKJ1hDjswtHYjUbGdoj7j1e8ihDbPoE2qOoID7vUBDjq6Xu8O21ctvSTYu0RAVyZV6uPa7xlgtKeWRFMn9zaZLKR1VfvYHN43DQaB7JncadrsUuTr1aTjhMwaYa9o86XR3iCYzDVbIbURXqXu+Fe4sJ61SafKDJIGN9cqhinWOf2ZheKlM9VQ3UZyE16DW9axBqpFpj397lk0JmG9e86Eem1DWJNdZqR9bKchJRRllI3GMcNbY+x02B8N/Uh6QMCPCxbm0Y1nL5rfYYk2VSoSuOwzgsTx5IEoeDS2G9DDaJGYyY+llPfVNcURCjKvYc6QFr4aL7NbKCt7bOnG9ip0l0d34XyznPLN9tDyWFOGEXHEHkG96R9CosEWWyPjbA/hTonHR4dIn1CAd+a2B+lHDrZBTJG74YbTOgkjdpMlsDk6h2sFzBRvbSF9YkI1MKJpFLa+xTlIeSfYVGECbiJjUJT2yB1hrD84dpWGt4izko0VhDx1HsDJjFf2uJ2Bpd81csKWmBysHh5sKroGs2NOW0YhpdwM2PxBVsC4QrCHEXFzFs5lboEHebQ9IpJGwouAEjJ9CssWcutMWFCnBelwytelNDXFpCkJloRrMjct5H+1kDf0gdL8svP2E0fi/I6w1LIsjEaZo8TI7cTl750NaFIopm4NmqKjuatK5JzSwp05f+9NrW6fF/y/arFAlQt2S02SmhrXOg1/pftwVHipNAd4W73GEsAf2AZ8geQKI3hYJRJCjo2g+DxYYEuBBwC8zrLEs5BsI46bGmRwE5UmloZvQWefSBO5Mof71HStTE09tTmLYir04ceaxPicsawZYhvWB/C/gArya2QqpZMW5feo7HUHkE7jKZ+w2yMTzVOy3ml3wAgDWcIGd2eWwyCcksl2b8c6/3+/I2Z4BoK4cbXmVrnG/2CpMreRxKVkO7I+l3ll1EooD+WWczlMtgEI80tVdAHqbQj2NsB0jVegKry2AK6ra9SgCY3xBOgqvA948iggOyQJZBZ5CzSSvGUeX1fipJKpxP5RHo0SEb3ojXKQOnFKKWZ7+V3eidJp9Nfw++S25ok24aHX79oqFWB2WYhIFJoKWqpz4UYAnrwO2eabfZqbKKCaPca79G7BPhzoa1vS7a0qfMV5tVp79O+ByzPLftOU7SXBDt5SuM5q9S23+imZfFIerUhgK4zC/8qoGVWKVGT8hUsqcixlDZMn4wmMdAezwmQUKaQAkwyQCXgTQJdSSQiNJeADFxYyilR1xc+6Yr6Idr3HXEj9ijmNEmS3EpFOd9Yw6LN8RRY9ws6UqCyi+FNcAiiEg6FDZZa/LUroPgkQJBjyONIllEi9oqADC1GgeItcm1yqbCrnQhg0WPAXcPHWBN+Nk+pP8ixiPNSlML+vkjJ+Ryr3B2z9PP4NhZqGE3/CobjnLupaI14pt87D5SqcQKSgayEciIPKMioStlEBXxzyEg3soP05thQSyyJsb+uUI5k/HIk1VLLAI/qwwmMykkwe/rvDw89IQFPWlErJJBivXRsbjqHgAQgfmBFIw4AQI8q0EyV3mPHSOFdLlGNVKSV2XnCjQ/SbSoSzsY5AUNWfMjf4caSinjiP1PDGgbVp4NYyakY8hwLoKXzavKG9LifTtLQCJVsHzbNsFUd5FmjZwj7eG+8Ou2CE1PUALPaaLx877AVeCM0oY9chhp6k/0D+9duBIxgMefwZ3SUiZAXgcCBWX9D2a8RWnrNe29g76wg01LqkRcY2EqRQ9a6w/4U0iLWiEAEtCeqX94ZOQKHafhXhdabw5FSGrckh0o9tjijMgVhgmE1YIpT2xsKoOdjtoO7L5W2BvFXp6Ra/X/gEtHj0t4uhAFvs+S1h/Ax3m/Z6ZSxU//Y7OHCevfctk7RjQcWORdEY2Rh6I8vjssCGaGaMmibRXSENFN3ZLE4QuwYQ8r/63YsX9soYvThLEA0pA3XkAAFvbAXN7ZO027erUlXrQRmXhm3DaW5FB7cxyrButnJfOzIqY+5iFkz4x9KvqVOaMzPgURP6RHK9FQVcuG/UqErQ5VHkiMF3xMiSasrpwOx4qAd8tJPhHWpOTatoMZFC8k9zw3Dr/C/i96xD2fUT0xQLK2tnEGLF876MFEVAYCi/ua3rjAalDu2F+yLl2L0WwxZMB4/aQFqNXtM5d++2IRp6dbhFOkw1S8pi6hpMIBYbipU0h9CCbeY9jUfq2w4kUvU+iQFquKP3Yq+np1Y/dUeLEbZF7HWsT9d6NNmn3RIavZm8R7BESLEY+YUsVQvvAlFrP+Fl+9qxMGq7zTd2Wbf5xiyWputO5jsdQGm9Ufun4wUFiTqaeRVys7l94cMAabpBtXPVA7TVV1pniEPc25UpjWJ3nEcirOqpu3EHblUuRt+5dVBToBHB0HWVg3WI11Dqz+2/ZyJCMoPt9UMnRIujaFEHWdh9F1GBSfVFausTwx00XQQVm7UmzwoP9YaG/hUuOAfiDilS7nGS4XBPKOHTwPXYzow12c7/x2n68OjzFkzd316u1WL1oslrKRoqAYtzERhV3kwPPpWXW9zZ1BWXkaAVqPShvpjk2RvQ71vvMwkqtia/3ugYKeCueffdc5AU+U2Kmtu7nyY5gipRNJVDJ4Xme2tnNYd91t4igfeM2gCNojH/WooDKiSnU5z2C4lsnSZGjLmFASte8Yya0FEiPqK6cxns758zuP3IYUfk/fN6FXP8ZyDo2CONrk/ITHtZ5hkTqmJNVzXKWlMQfxzOKpUNoUPDsITrM6R9S6y+PlW/U/asZ+IZOi/MR3VVrHcadHtaXiXmJZLiPrFDBds5j6RwqsUEkPv6hDsjOpdMCpaMPvMWYXPbqck2weoJ+42qN/JpZHnTVNeBzfwJr30i7Up/pUFYaO9DOz0cB996qqn4kr0v62JxrbNa82A500Yor51U1bFEvwgaTXsK10x60WNmU1t6Eouv6VUNDjd+awCuSpThLKb4zFXlBXE8JEcVFvN4gXqAwrqVKaeuJLO+tm7ayFvuhHpbrIZGsUEnTGcE2iwWFUhUUZ5Jl5Hp081Uetlgkd5CLRw9zFu4F+0b/KpcWiISXD15I/y2Bm5dKCdC2lLMrcgyL4RslHQ2nPk6YTZBraRZXqifRXJQVVI+GtlQkazAkPKmvSq6lI+e/4wIOpHz+LwYuYa0xX0yaLj2dZWnubObed9GLgCdrOpZXmVaL/KwXiXLtfQbxZ9qGab8caaS1uJlK0Il+KdSfa34jeaIdrIdsT+3Q0remR1kqkMxjvbEkGxHrtg1Bs+2/IemaBjhHc9yA0x5sYG0S1CLkAQwNIpxl9sTsA0mgsqZlHLpBtQxPb2fdWCmtd3rkWQM5KB5R90zeg5V/CRQl4oOyL5o6BgV9+jXld36UiUbFDnP1Jf2JAwW9CauWqIxlAiGLWk5aDYY3qrhyz0RFpS3TjEPEVN1KZ067FA1AYpt8F59gWMMveapj3CM8c4fx0XG3QaEPk2PRDpfQJ2JT7oE5i3vIFv03/x327lXun6sC83bPhCdiyQalOY1TR5YhSEv2iozr8tu8schDh/Ya5+tCRAOabVPLJPdRG6nQU0rp7tvwxIyetu+bndvG8Ya+xm+x6Tf8WhM3Qgc49HvFA0TxtYFhj6aWZiVQurpqpVUoyfXUr39uObqGE8EmVQNAK8ukdaCrdqlZ+v35/ANEmpDo6uH0hhaRbV6Uayzg8O27kAvu8GEFJMHyMOjKQ7CBZ/8fx3K76lDeWYkhBtYBtOpIrpVnOGirKDABVvcBKwoXNT6EhS/3S8rn3ARXwgxbEpcUMPN/8YVf7dBo6ztChNy3+9l1K/gX4waO0eqTladYFA+2PoJP3Tpli1ddptTwh53CRACcJvFHbX9pkulbxp2iLHPh41wpD90vHovD+2QpHzJN+zQDEr6LS+bn1RTLemzKREza+cJpeqX0Qz1aZTfp/ybqbOGquEUNMG79b2tWR+bwVbpj49Xn/DJIE8659ySoWBhTFbCJ8OwMXsKe4yHbKzv/Iyp7gfONAcFlpTqT4IQdRHxgZBefmEtLQ/2h9xpCmC4iTENSptRIPPzPToezqtIzCuR+CgY9CeFeNXnwcan4BUPEMgwLzVSLl8DHObLijbCGX+DwL/4ihKlNYJgnkX41I3RcxzO5zQM7+K5oxFU50gG/ob860AHPHxNN25DhOsPjO/D9OJRJ/NIGHK9uPgHXHqxcSQjHQZ+Gpv+HMwTRIovuhmHdz9u44swvYjNbdyDTn+ZpX9J6k3ftyDyCy1PoJIqohcRfajlCTQkqk+hoS+29KJRynukT3ATZW9XoYLb4OaV2Xxs7iK5sSj0JrZU7SQRgL5nJ6Fr9WA6x43slJ2aOqCu4/p5aHqohpkq4Oy2GtY2SgWs3TbsihlzJaYwt2zWuot9qIgOum1oHBPG6SInJptMIH3OKBBqGDYOGwuAa9HthdOFAdz3eTTDdRXA0r0z1CrYAFn5oyqc5IY+IxbQIYVPOFI2PQhIioJAJriESA3+F5H6FH1tUwAA", "sha256": "2e9b9f47f51bc3571366883311f1f9200f839335c06aaa7c6e16bf7c602c24de"}}')
SNAPSHOT_ROOT.mkdir(parents=True, exist_ok=True)
for name, item in embedded.items():
    raw = gzip.decompress(base64.b64decode(item["payload"]))
    if hashlib.sha256(raw).hexdigest() != item["sha256"]:
        raise ValueError(f"Embedded file hash mismatch: {name}")
    target = SNAPSHOT_ROOT / name
    if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest() != item["sha256"]:
        raise ValueError(f"Existing frozen source snapshot differs: {target}")
    target.write_bytes(raw)

if str(SNAPSHOT_ROOT) not in sys.path:
    sys.path.insert(0, str(SNAPSHOT_ROOT))
for module_name in (
    "cfpb_v052_pipeline_smoke_v02_common",
    "prepare_cfpb_seed_v052_pipeline_smoke_v02",
    "validate_cfpb_v052_pipeline_smoke_v02",
    "run_cfpb_v052_blind_semantic_judge_v01",
    "run_cfpb_v052_blind_semantic_judge_v02",
    "evaluate_cfpb_v052_semantic_judge_v01",
    "freeze_cfpb_v052_semantic_judge_v01",
):
    sys.modules.pop(module_name, None)
from cfpb_v052_pipeline_smoke_v02_common import sha256_file, load_semantic_judgments
from run_cfpb_v052_blind_semantic_judge_v01 import (
    fetch_zdr_endpoint_preflight,
    load_blind_cases,
)
from run_cfpb_v052_blind_semantic_judge_v02 import (
    RunnerConfig,
    build_evidence_units,
    run_judge,
)
from validate_cfpb_v052_pipeline_smoke_v02 import validate_and_route
from evaluate_cfpb_v052_semantic_judge_v01 import evaluate_judge
from freeze_cfpb_v052_semantic_judge_v01 import freeze_judge

PROMPT = SNAPSHOT_ROOT / "cfpb_v052_semantic_judge_v02_attempt02.md"
RUNNER_SOURCE = SNAPSHOT_ROOT / "run_cfpb_v052_blind_semantic_judge_v02.py"
COMMON_SOURCE = SNAPSHOT_ROOT / "cfpb_v052_pipeline_smoke_v02_common.py"
VALIDATOR_SOURCE = SNAPSHOT_ROOT / "validate_cfpb_v052_pipeline_smoke_v02.py"
print({name: item["sha256"] for name, item in embedded.items()})


## 3. Inspect payload, verify the live ZDR endpoint, and provide the API key


In [ ]:
from getpass import getpass
import pandas as pd

cases = load_blind_cases(RAW, PREPARED)
assert len(cases) == 20
assert all(
    set(case.model_dump()) == {"seed_id", "labels", "grounding", "generated"}
    for case in cases
)
preview_units = build_evidence_units(cases[0])
print({
    "rows": len(cases),
    "fields_visible_to_judge": ["case_id", "evidence_units"],
    "evidence_protocol": "stable_reference_ids_v01",
    "oracle_loaded_by_runner": False,
    "candidate_judgments_loaded_by_runner": False,
    "prompt_sha256": sha256_file(PROMPT),
    "first_case_evidence_units": len(preview_units),
})
display(pd.DataFrame([
    {
        "ref_id": unit.ref_id,
        "source": unit.source,
        "location": unit.location,
        "text_preview": unit.text[:180],
    }
    for unit in preview_units[:10]
]))

CONFIG = RunnerConfig(
    model_name="qwen/qwen3.5-35b-a3b",
    base_url="https://openrouter.ai/api/v1",
    judge_id=(
        "openrouter_qwen3_5_35b_a3b_deepinfra_zdr_"
        "nonthinking_blind_v02_attempt02"
    ),
    sampling_profile="qwen3.5_official_instruct_general",
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    min_p=0.0,
    presence_penalty=1.5,
    repetition_penalty=1.0,
    max_tokens=2048,
    provider_slug="deepinfra",
    allow_fallbacks=False,
    require_parameters=True,
    data_collection="deny",
    zdr=True,
    reasoning_enabled=False,
    reasoning_exclude=True,
    seed=20260722,
    max_attempts=2,
    request_timeout_seconds=120.0,
)
print(CONFIG.model_dump())

endpoint_preflight = fetch_zdr_endpoint_preflight(
    config=CONFIG,
    output_path=ENDPOINT_PREFLIGHT,
)
print({
    "zdr_endpoint_preflight": "passed",
    "provider": endpoint_preflight.selected_endpoint.provider_name,
    "tag": endpoint_preflight.selected_endpoint.tag,
    "status": endpoint_preflight.selected_endpoint.status,
    "supported_parameters": endpoint_preflight.selected_endpoint.supported_parameters,
})

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass(
        "OPENROUTER_API_KEY (not persisted): "
    )
if not os.environ["OPENROUTER_API_KEY"].strip():
    raise RuntimeError("OPENROUTER_API_KEY is empty")


## 4. Two-row reference-contract smoke

These fixed development IDs exercise one previously accepted row and one
semantic-reject row. Expected labels are not sent or loaded. The model may
return only reference IDs that exist in the supplied evidence-unit table.
A second call is permitted only for a non-empty JSON/schema/reference
contract failure.


In [ ]:
CONTRACT_IDS = [
    "cfpb_v052_0d210ece132c66d3",
    "cfpb_v052_2c70d920e4cfccd5",
]
contract_report = run_judge(
    raw_path=RAW,
    prepared_path=PREPARED,
    prompt_path=PROMPT,
    judgments_path=CONTRACT_ROOT / "semantic_judgments.jsonl",
    raw_responses_path=CONTRACT_ROOT / "raw_responses.jsonl",
    failed_attempts_path=CONTRACT_ROOT / "failed_attempts.jsonl",
    endpoint_preflight_path=ENDPOINT_PREFLIGHT,
    report_path=CONTRACT_ROOT / "judge_run_report.json",
    config=CONFIG,
    api_key=os.environ["OPENROUTER_API_KEY"],
    selected_seed_ids=CONTRACT_IDS,
)
assert contract_report["semantic_coverage_complete"] is True
contract = load_semantic_judgments(CONTRACT_ROOT / "semantic_judgments.jsonl")
display(pd.DataFrame([
    {"seed_id": key, "decision": value.decision, "reasons": value.reasons}
    for key, value in sorted(contract.items())
]))


## 5. Run/resume the blind 20-row development calibration


In [ ]:
import shutil

# Reuse the two contract-smoke judgments only when starting a fresh
# calibration. They used the exact same prompt, config, and schema;
# their separate originals remain as contract evidence.
calibration_judgments = CALIBRATION_ROOT / "semantic_judgments.jsonl"
calibration_responses = CALIBRATION_ROOT / "raw_responses.jsonl"
calibration_failures = CALIBRATION_ROOT / "failed_attempts.jsonl"
if not calibration_judgments.exists() and not calibration_responses.exists():
    CALIBRATION_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CONTRACT_ROOT / "semantic_judgments.jsonl", calibration_judgments)
    shutil.copy2(CONTRACT_ROOT / "raw_responses.jsonl", calibration_responses)
    contract_failures = CONTRACT_ROOT / "failed_attempts.jsonl"
    if contract_failures.exists():
        shutil.copy2(contract_failures, calibration_failures)
    print("Bootstrapped the fresh calibration cache with 2 contract-smoke rows")
elif calibration_judgments.exists() != calibration_responses.exists():
    raise ValueError("Calibration judgment/audit cache is incomplete")

calibration_report = run_judge(
    raw_path=RAW,
    prepared_path=PREPARED,
    prompt_path=PROMPT,
    judgments_path=calibration_judgments,
    raw_responses_path=calibration_responses,
    failed_attempts_path=calibration_failures,
    endpoint_preflight_path=ENDPOINT_PREFLIGHT,
    report_path=CALIBRATION_ROOT / "judge_run_report.json",
    config=CONFIG,
    api_key=os.environ["OPENROUTER_API_KEY"],
)
assert calibration_report["semantic_coverage_complete"] is True
print({
    "rows": calibration_report["judgment_rows"],
    "new_provider_calls": calibration_report["new_provider_calls"],
    "decision_counts": calibration_report["decision_counts"],
    "failed_attempt_rows": calibration_report["outputs"]["failed_attempts"]["rows"],
})


## 6. Score v02 only after all blind judgments exist

This is the first cell that loads the versioned GPT-5.6-SOL development
adjudication v02. It measures agreement, not human accuracy. The v02
oracle differs from v01 by one independently reviewed reason addition.


In [ ]:
EVALUATED = CALIBRATION_ROOT / "evaluated"
evaluation = validate_and_route(
    raw_output=RAW,
    prepared_input=PREPARED,
    input_manifest=INPUT_MANIFEST,
    disposition_path=PRIVACY_DISPOSITION,
    validated_path=EVALUATED / "validated/dialogues.jsonl",
    rejected_path=EVALUATED / "rejected/dialogues.jsonl",
    review_path=EVALUATED / "review/dialogues.jsonl",
    report_path=EVALUATED / "validation_report_v02.json",
    semantic_judgments_path=CALIBRATION_ROOT / "semantic_judgments.jsonl",
    oracle_path=ORACLE,
)
metrics = evaluation["oracle_metrics"]
calibration_metrics = evaluate_judge(
    judgments_path=CALIBRATION_ROOT / "semantic_judgments.jsonl",
    oracle_path=ORACLE,
    combined_report_path=EVALUATED / "validation_report_v02.json",
    output_path=CALIBRATION_ROOT / "calibration_metrics_v02.json",
)
display(pd.DataFrame({
    "judge_only": pd.Series(calibration_metrics["judge_only_metrics"]),
    "combined_validator": pd.Series(calibration_metrics["combined_validator_metrics"]),
}))
print({
    "validated": evaluation["validated_rows"],
    "rejected": evaluation["rejected_rows"],
    "review": evaluation["review_rows"],
    "development_only": True,
    "human_gold": False,
})


# Automatic development gates are necessary but not sufficient for freezing.
# They require attempt02 to exceed attempt01 and roughly match or exceed the
# earlier candidate-prefill development baseline. Human review remains separate.
FREEZE_THRESHOLDS = {
    "decision_reject_precision": 0.95,
    "decision_reject_recall": 0.80,
    "decision_reject_f1": 0.85,
    "exact_set_accuracy": 0.65,
    "micro_precision": 0.75,
    "micro_recall": 0.60,
    "micro_f1": 0.70,
}
judge_metrics = calibration_metrics["judge_only_metrics"]
FREEZE_METRIC_CHECKS = {
    name: float(judge_metrics[name]) >= threshold
    for name, threshold in FREEZE_THRESHOLDS.items()
}
AUTO_FREEZE_METRICS_PASSED = all(FREEZE_METRIC_CHECKS.values())
display(pd.DataFrame([
    {
        "metric": name,
        "observed": float(judge_metrics[name]),
        "threshold": FREEZE_THRESHOLDS[name],
        "passed": passed,
    }
    for name, passed in FREEZE_METRIC_CHECKS.items()
]))
print({"automatic_freeze_metrics_passed": AUTO_FREEZE_METRICS_PASSED})


## 7. Apply the explicit attempt02 freeze gate

Freezing requires all automatic metric thresholds, a real human review
of the development disagreements, an explicit calibration-review flag,
and a separate approval flag with an approver ID. The independent
GPT-5.6-SOL second pass is not a substitute for the human-review flag.


In [ ]:
DEVELOPMENT_DISAGREEMENTS_REVIEWED_BY_HUMAN = False
CALIBRATION_REVIEW_COMPLETED = False
APPROVE_FREEZE = False
APPROVER_ID = ""

freeze_blockers = []
if not AUTO_FREEZE_METRICS_PASSED:
    freeze_blockers.append("automatic metric thresholds failed")
if not DEVELOPMENT_DISAGREEMENTS_REVIEWED_BY_HUMAN:
    freeze_blockers.append("development disagreements lack human review")
if not CALIBRATION_REVIEW_COMPLETED:
    freeze_blockers.append("calibration review is not marked complete")
if not APPROVE_FREEZE:
    freeze_blockers.append("freeze approval is false")
if not APPROVER_ID.strip():
    freeze_blockers.append("approver ID is empty")

if freeze_blockers:
    print({"frozen": False, "blockers": freeze_blockers})
else:
    frozen = freeze_judge(
        judge_report_path=CALIBRATION_ROOT / "judge_run_report.json",
        judgments_path=CALIBRATION_ROOT / "semantic_judgments.jsonl",
        evaluation_path=EVALUATED / "validation_report_v02.json",
        metrics_path=CALIBRATION_ROOT / "calibration_metrics_v02.json",
        prompt_path=PROMPT,
        oracle_path=ORACLE,
        raw_path=RAW,
        prepared_path=PREPARED,
        runner_path=RUNNER_SOURCE,
        common_path=COMMON_SOURCE,
        validator_path=VALIDATOR_SOURCE,
        endpoint_preflight_path=ENDPOINT_PREFLIGHT,
        output_path=(
            REVALIDATION_ROOT
            / "judges/frozen/cfpb_v052_semantic_judge_v02_attempt02.json"
        ),
        approved_by=APPROVER_ID,
        approve_freeze=True,
    )
    print(json.dumps(frozen.model_dump(mode="json"), ensure_ascii=False, indent=2))


## Stop point

If any automatic or human gate is blocked, preserve attempt02 as
development lineage and do not run a held-out smoke. If the judge is
genuinely frozen, the next notebook must generate a new held-out
pipeline-only smoke and apply the frozen manifest without calibration on
held-out answers. A finite human audit follows that fixed evaluation.
Formal 50–100-row generation remains blocked until a separately approved
privacy protocol makes `benchmark_release_gate_passed=true`.
